# 西雅图住房价格GCCM因果结构实验

## 实验目的

本实验在已经完成的CP、GeoCP、GeoSIMCP-EUC和GeoSIMCP-MND基线实验基础上，使用静态空间数据的GCCM方法提取住房属性与住房价格之间的因果信息。

GCCM结果将用于构造后续GCCP中的因果距离。为保证与Baseline公平比较，GCCP继续使用：

- 相同的西雅图住房数据；
- 相同的Train、Calibration和Test索引；
- 相同的Train内部调参索引；
- 相同的XGBoost基础预测模型；
- 相同的目标覆盖率和评价指标。

变量按照实验角色划分为三类：

1. **预测目标变量**：住房价格 `price`；
2. **候选因果属性变量**：8个非空间住房属性；
3. **空间定位变量**：`UTM_X`和`UTM_Y`，用于构建静态空间邻域。

为避免信息泄漏，GCCM因果信息不使用外层Calibration和Test估计。

在Train内部调参阶段，使用Inner Fit估计临时GCCM因果信息；在最终实验阶段，使用完整外层Train重新估计最终GCCM因果信息。

## 1. 数据读取与固定实验划分

本节读取西雅图住房数据，并恢复Baseline实验已经保存的外层三段划分和Train内部调参划分。

GCCM阶段不重新随机划分数据，而是继续使用：

- Outer Train：2100条；
- Outer Calibration：450条；
- Outer Test：450条；
- Inner Fit：1470条；
- Inner Calibration：315条；
- Inner Validation：315条。

GCCM将在两个阶段分别估计：

- 使用Inner Fit估计临时因果结构，供GCCP内部参数选择使用；
- 使用完整Outer Train估计最终因果结构，供最终GCCP实验使用。

Outer Calibration和Outer Test不参与GCCM因果结构估计。

In [2]:
# ============================================================
# 第1章：读取数据并恢复Baseline固定划分
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. 设置项目路径
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    r"E:\GeoSIMCP_Reproduce\geoconformal-main"
)

DATA_PATH = (
    PROJECT_ROOT
    / "example"
    / "seattle_sample_3k.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "outputs"
)

SPLIT_PATH = (
    OUTPUT_DIR
    / "seattle_split_seed42.npz"
)

INNER_SPLIT_PATH = (
    OUTPUT_DIR
    / "seattle_inner_split_seed42.npz"
)

GCCM_OUTPUT_DIR = (
    OUTPUT_DIR
    / "gccm"
)

GCCM_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 2. 检查必要文件
# ------------------------------------------------------------

required_paths = [
    DATA_PATH,
    SPLIT_PATH,
    INNER_SPLIT_PATH
]

for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(
            f"找不到必要文件：{path}"
        )


# ------------------------------------------------------------
# 3. 读取原始住房数据
# ------------------------------------------------------------

data = pd.read_csv(DATA_PATH)

data["price"] = (
    np.power(
        10,
        data["log_price"]
    )
    / 10000
)


# ------------------------------------------------------------
# 4. 定义三类变量
# ------------------------------------------------------------

ASpatial_FEATURES = [
    "bathrooms",
    "sqft_living",
    "sqft_lot",
    "grade",
    "condition",
    "waterfront",
    "view",
    "age"
]

SPATIAL_COORDINATES = [
    "UTM_X",
    "UTM_Y"
]

TARGET_VARIABLE = "price"


# ------------------------------------------------------------
# 5. 读取外层三段划分
# ------------------------------------------------------------

outer_split = np.load(
    SPLIT_PATH
)

train_indices = outer_split[
    "train_indices"
]

calibration_indices = outer_split[
    "calibration_indices"
]

test_indices = outer_split[
    "test_indices"
]


# ------------------------------------------------------------
# 6. 读取Train内部调参划分
# ------------------------------------------------------------

inner_split = np.load(
    INNER_SPLIT_PATH
)

inner_fit_indices = inner_split[
    "inner_fit_indices"
]

inner_calibration_indices = inner_split[
    "inner_calibration_indices"
]

inner_validation_indices = inner_split[
    "inner_validation_indices"
]


# ------------------------------------------------------------
# 7. 构造GCCM所需数据
# ------------------------------------------------------------

X_causal_all = data[
    ASpatial_FEATURES
].copy()

coordinates_all = data[
    SPATIAL_COORDINATES
].copy()

y_all = data[
    TARGET_VARIABLE
].copy()


# Inner Fit：临时GCCM因果结构
X_causal_inner_fit = X_causal_all.iloc[
    inner_fit_indices
].copy()

coordinates_inner_fit = coordinates_all.iloc[
    inner_fit_indices
].copy()

y_inner_fit = y_all.iloc[
    inner_fit_indices
].copy()


# Outer Train：最终GCCM因果结构
X_causal_outer_train = X_causal_all.iloc[
    train_indices
].copy()

coordinates_outer_train = coordinates_all.iloc[
    train_indices
].copy()

y_outer_train = y_all.iloc[
    train_indices
].copy()


# ------------------------------------------------------------
# 8. 检查数据划分和信息隔离
# ------------------------------------------------------------

train_set = set(
    train_indices.tolist()
)

calibration_set = set(
    calibration_indices.tolist()
)

test_set = set(
    test_indices.tolist()
)

inner_fit_set = set(
    inner_fit_indices.tolist()
)

inner_calibration_set = set(
    inner_calibration_indices.tolist()
)

inner_validation_set = set(
    inner_validation_indices.tolist()
)


assert len(data) == 3000

assert len(train_indices) == 2100
assert len(calibration_indices) == 450
assert len(test_indices) == 450

assert len(inner_fit_indices) == 1470
assert len(inner_calibration_indices) == 315
assert len(inner_validation_indices) == 315

assert (
    inner_fit_set
    | inner_calibration_set
    | inner_validation_set
) == train_set

assert train_set.isdisjoint(
    calibration_set
)

assert train_set.isdisjoint(
    test_set
)

assert calibration_set.isdisjoint(
    test_set
)

assert X_causal_inner_fit.shape == (
    1470,
    8
)

assert coordinates_inner_fit.shape == (
    1470,
    2
)

assert X_causal_outer_train.shape == (
    2100,
    8
)

assert not X_causal_inner_fit.isna().any().any()
assert not coordinates_inner_fit.isna().any().any()
assert not y_inner_fit.isna().any()


# ------------------------------------------------------------
# 9. 输出变量角色
# ------------------------------------------------------------

variable_roles = pd.DataFrame({
    "Role": [
        "Target",
        "Candidate causal attributes",
        "Spatial coordinates"
    ],
    "Variables": [
        TARGET_VARIABLE,
        ", ".join(ASpatial_FEATURES),
        ", ".join(SPATIAL_COORDINATES)
    ],
    "Usage": [
        "GCCM被解释变量",
        "估计住房属性与价格之间的因果信息",
        "构造静态空间邻域"
    ]
})

display(variable_roles)


# ------------------------------------------------------------
# 10. 输出检查结果
# ------------------------------------------------------------

print("数据总量：", len(data))
print("Outer Train：", len(train_indices))
print("Outer Calibration：", len(calibration_indices))
print("Outer Test：", len(test_indices))

print("\nInner Fit：", len(inner_fit_indices))
print("Inner Calibration：", len(inner_calibration_indices))
print("Inner Validation：", len(inner_validation_indices))

print(
    "\nInner Fit未使用Outer Calibration：",
    inner_fit_set.isdisjoint(calibration_set)
)

print(
    "Inner Fit未使用Outer Test：",
    inner_fit_set.isdisjoint(test_set)
)

print(
    "\nGCCM输出目录：",
    GCCM_OUTPUT_DIR
)

print(
    "\n第1章完成："
    "数据和固定实验划分已经恢复。"
)

,Role,Variables,Usage
0,Target,price,GCCM被解释变量
1,Candidate causal attributes,"bathrooms, sqft_living, sqft_lot, grade, condi...",估计住房属性与价格之间的因果信息
2,Spatial coordinates,"UTM_X, UTM_Y",构造静态空间邻域


数据总量： 3000
Outer Train： 2100
Outer Calibration： 450
Outer Test： 450

Inner Fit： 1470
Inner Calibration： 315
Inner Validation： 315

Inner Fit未使用Outer Calibration： True
Inner Fit未使用Outer Test： True

GCCM输出目录： E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccm

第1章完成：数据和固定实验划分已经恢复。


## 2. GCCM实现与运行环境检查

在正式计算静态空间GCCM之前，本节检查：

1. 当前Python环境中是否已经安装CCM或GCCM相关工具；
2. 项目目录中是否存在以前编写或下载的GCCM代码；
3. `02_gccm`和`03_gccp`文件夹中是否已经保存了相关实验文件。

需要注意，普通Convergent Cross Mapping通常面向时间序列，而本实验需要处理静态空间住房数据。因此，即使环境中安装了CCM包，也必须进一步检查其是否支持基于空间邻域的GCCM，不能直接套用。

In [3]:
# ============================================================
# 第2章：检查现有GCCM实现与相关工具
# ============================================================

import importlib.util


# ------------------------------------------------------------
# 1. 检查当前Python环境中的CCM相关模块
# ------------------------------------------------------------

candidate_modules = [
    "pyEDM",
    "skccm",
    "causal_ccm",
    "gccm"
]

module_records = []

for module_name in candidate_modules:

    try:
        module_spec = importlib.util.find_spec(
            module_name
        )

        installed = module_spec is not None

        module_location = (
            module_spec.origin
            if installed
            else None
        )

    except Exception as error:

        installed = False
        module_location = f"检查失败：{error}"

    module_records.append({
        "Module": module_name,
        "Installed": installed,
        "Location": module_location
    })


module_status = pd.DataFrame(
    module_records
)

print("当前环境中的CCM相关模块：")

display(module_status)


# ------------------------------------------------------------
# 2. 查看GCCM和GCCP实验文件夹中已有文件
# ------------------------------------------------------------

GCCM_NOTEBOOK_DIR = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "02_gccm"
)

GCCP_NOTEBOOK_DIR = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "03_gccp"
)


def list_folder_files(folder_path):
    """
    列出指定文件夹下的文件和子文件夹。
    """
    records = []

    if not folder_path.exists():
        return pd.DataFrame(
            columns=[
                "Name",
                "Type",
                "Relative path",
                "Size KB"
            ]
        )

    for item_path in sorted(
        folder_path.rglob("*")
    ):

        try:
            relative_path = item_path.relative_to(
                PROJECT_ROOT
            )

            records.append({
                "Name": item_path.name,
                "Type": (
                    "Folder"
                    if item_path.is_dir()
                    else "File"
                ),
                "Relative path": str(relative_path),
                "Size KB": (
                    round(
                        item_path.stat().st_size / 1024,
                        2
                    )
                    if item_path.is_file()
                    else np.nan
                )
            })

        except Exception:
            continue

    return pd.DataFrame(records)


gccm_folder_contents = list_folder_files(
    GCCM_NOTEBOOK_DIR
)

gccp_folder_contents = list_folder_files(
    GCCP_NOTEBOOK_DIR
)


print("\n02_gccm文件夹内容：")

if len(gccm_folder_contents) > 0:
    display(gccm_folder_contents)
else:
    print("02_gccm文件夹为空。")


print("\n03_gccp文件夹内容：")

if len(gccp_folder_contents) > 0:
    display(gccp_folder_contents)
else:
    print("03_gccp文件夹为空。")


# ------------------------------------------------------------
# 3. 在项目的主要代码目录中搜索GCCM/CCM关键词
# ------------------------------------------------------------

search_roots = [
    PROJECT_ROOT / "GCCP_Seattle",
    PROJECT_ROOT / "geoconformal",
    PROJECT_ROOT / "Codes_Manuscript_GeoSIMCP"
]

search_keywords = [
    "gccm",
    "geographical convergent cross mapping",
    "convergent cross mapping",
    "cross mapping",
    "crossmap"
]

searchable_suffixes = {
    ".py",
    ".ipynb",
    ".r",
    ".md",
    ".txt"
}

matching_records = []


for search_root in search_roots:

    if not search_root.exists():
        continue

    for file_path in search_root.rglob("*"):

        if not file_path.is_file():
            continue

        if file_path.suffix.lower() not in searchable_suffixes:
            continue

        try:
            # 跳过异常大的文本文件
            if file_path.stat().st_size > 10_000_000:
                continue

            file_text = file_path.read_text(
                encoding="utf-8",
                errors="ignore"
            )

            lower_text = file_text.lower()

            matched_keywords = [
                keyword
                for keyword in search_keywords
                if keyword in lower_text
            ]

            if matched_keywords:

                matching_records.append({
                    "File name": file_path.name,
                    "Relative path": str(
                        file_path.relative_to(
                            PROJECT_ROOT
                        )
                    ),
                    "Matched keywords": ", ".join(
                        matched_keywords
                    ),
                    "Size KB": round(
                        file_path.stat().st_size / 1024,
                        2
                    )
                })

        except Exception:
            continue


matching_code_files = pd.DataFrame(
    matching_records,
    columns=[
        "File name",
        "Relative path",
        "Matched keywords",
        "Size KB"
    ]
)


print("\n项目中包含GCCM或Cross Mapping关键词的文件：")

if len(matching_code_files) > 0:
    display(matching_code_files)
else:
    print("没有找到已有的GCCM或Cross Mapping代码。")


print(
    "\n第2章检查完成："
    "已检查Python模块、实验文件夹和项目代码。"
)

当前环境中的CCM相关模块：


,Module,Installed,Location
0,pyEDM,False,None
1,skccm,False,None
2,causal_ccm,False,None
3,gccm,False,None



02_gccm文件夹内容：


,Name,Type,Relative path,Size KB
0,.ipynb_checkpoints,Folder,GCCP_Seattle\02_gccm\.ipynb_checkpoints,NaN
1,02_Seattle_GCCM_causal_structure-checkpoint.ipynb,File,GCCP_Seattle\02_gccm\.ipynb_checkpoints\02_Sea...,1.97
2,02_Seattle_GCCM_causal_structure.ipynb,File,GCCP_Seattle\02_gccm\02_Seattle_GCCM_causal_st...,14.93



03_gccp文件夹内容：
03_gccp文件夹为空。

项目中包含GCCM或Cross Mapping关键词的文件：


,File name,Relative path,Matched keywords,Size KB
0,01_Seattle_baselines_three_split.ipynb,GCCP_Seattle\01_baselines\01_Seattle_baselines...,gccm,266.57
1,01_Seattle_baselines_three_split-checkpoint.ipynb,GCCP_Seattle\01_baselines\.ipynb_checkpoints\0...,gccm,266.57
2,02_Seattle_GCCM_causal_structure.ipynb,GCCP_Seattle\02_gccm\02_Seattle_GCCM_causal_st...,gccm,14.93
3,02_Seattle_GCCM_causal_structure-checkpoint.ipynb,GCCP_Seattle\02_gccm\.ipynb_checkpoints\02_Sea...,gccm,1.97
4,GeoSIMCP_simulation_0425.ipynb,Codes_Manuscript_GeoSIMCP\GeoSIMCP_simulation_...,gccm,5212.43
5,GeoSIMCP_synthetic_data_validation.ipynb.ipynb,Codes_Manuscript_GeoSIMCP\GeoSIMCP_synthetic_d...,gccm,2846.10



第2章检查完成：已检查Python模块、实验文件夹和项目代码。


### 2.1 GCCM环境检查结论

当前Python环境中未安装GCCM或CCM相关模块，项目的 `03_gccp`
文件夹中也不存在已有GCCP代码。

项目关键词搜索主要返回Baseline实验和当前GCCM Notebook，原因是
这些文件的说明文字中出现了GCCM。作者的
`GeoSIMCP_simulation_0425.ipynb` 同样出现GCCM关键词，需要进一步
检查其中是否包含可复用的算法实现。

本实验暂不安装普通时间序列CCM工具。西雅图住房数据属于不规则
空间点数据，后续应根据住房点之间的空间邻接关系构造空间滞后，
再实施静态空间GCCM。

老师推荐的SGWR论文不提供因果识别方法，但其将地理邻近性与属性
相似性共同纳入权重矩阵的思想，可用于后续GCCP联合距离的设计。

In [4]:
# ============================================================
# 第2章第2步：检查作者模拟Notebook中的GCCM相关内容
# ============================================================

import json


# ------------------------------------------------------------
# 1. 定位作者模拟Notebook
# ------------------------------------------------------------

simulation_matches = list(
    (
        PROJECT_ROOT
        / "Codes_Manuscript_GeoSIMCP"
    ).rglob(
        "GeoSIMCP_simulation_0425.ipynb"
    )
)

if len(simulation_matches) == 0:
    raise FileNotFoundError(
        "没有找到GeoSIMCP_simulation_0425.ipynb"
    )

SIMULATION_NOTEBOOK_PATH = simulation_matches[0]

print("找到作者模拟Notebook：")
print(SIMULATION_NOTEBOOK_PATH)


# ------------------------------------------------------------
# 2. 读取Notebook的JSON结构
# ------------------------------------------------------------

with open(
    SIMULATION_NOTEBOOK_PATH,
    "r",
    encoding="utf-8"
) as notebook_file:

    simulation_notebook = json.load(
        notebook_file
    )


# ------------------------------------------------------------
# 3. 搜索包含GCCM或Cross Mapping的单元格
# ------------------------------------------------------------

search_terms = [
    "gccm",
    "cross mapping",
    "cross-mapping",
    "crossmap",
    "causal distance"
]

matched_cells = []

for cell_number, cell in enumerate(
    simulation_notebook.get("cells", [])
):

    cell_source = "".join(
        cell.get("source", [])
    )

    lower_source = cell_source.lower()

    matched_terms = [
        term
        for term in search_terms
        if term in lower_source
    ]

    if matched_terms:

        matched_cells.append({
            "Cell number": cell_number,
            "Cell type": cell.get(
                "cell_type",
                "unknown"
            ),
            "Matched terms": ", ".join(
                matched_terms
            ),
            "Source length": len(cell_source)
        })

        print("\n" + "=" * 90)
        print("单元格编号：", cell_number)
        print("单元格类型：", cell.get("cell_type"))
        print("匹配关键词：", matched_terms)
        print("=" * 90)

        # 避免一次输出过长
        print(cell_source[:5000])

        if len(cell_source) > 5000:
            print("\n……该单元格后续内容已省略……")


# ------------------------------------------------------------
# 4. 汇总
# ------------------------------------------------------------

matched_cells_df = pd.DataFrame(
    matched_cells
)

print("\nGCCM相关单元格汇总：")

if len(matched_cells_df) > 0:
    display(matched_cells_df)
else:
    print("没有找到GCCM相关单元格。")

print(
    "\n匹配单元格数量：",
    len(matched_cells_df)
)

print(
    "\n第2章第2步完成："
    "作者模拟Notebook中的GCCM内容已经提取。"
)

找到作者模拟Notebook：
E:\GeoSIMCP_Reproduce\geoconformal-main\Codes_Manuscript_GeoSIMCP\GeoSIMCP_simulation_0425.ipynb

GCCM相关单元格汇总：
没有找到GCCM相关单元格。

匹配单元格数量： 0

第2章第2步完成：作者模拟Notebook中的GCCM内容已经提取。


### 2.2 作者GeoSIMCP项目检查结论

对作者的 `GeoSIMCP_simulation_0425.ipynb` 进行逐单元格搜索后，没有发现GCCM、Cross Mapping或Causal Distance相关代码。

因此，可以确认：

- GeoSIMCP作者代码不包含GCCM实现；
- GCCM不是GeoSIMCP代码中的一个现成组件；
- 后续需要单独参考GCCM原始论文及其官方代码；
- 暂不使用普通时间序列CCM工具替代空间GCCM。

下一步先检查本机是否具备运行GCCM官方代码所需的Git和R环境，再读取官方R代码，确认其输入数据、空间邻域和输出因果强度的具体实现。

在确认官方实现前，不直接自行编写住房点数据的GCCM，以避免与原始方法产生未经说明的差异。

In [6]:
# ============================================================
# 第2章第3步：检查Git和R运行环境
# ============================================================
import sys
import shutil
import subprocess
import platform

tool_candidates = {
    "Git": [
        "git",
        "git.exe"
    ],
    "R": [
        "R",
        "R.exe"
    ],
    "Rscript": [
        "Rscript",
        "Rscript.exe"
    ]
}

tool_records = []


# ------------------------------------------------------------
# 1. 查找工具安装路径
# ------------------------------------------------------------

for tool_name, candidate_names in tool_candidates.items():

    detected_path = None

    for candidate_name in candidate_names:

        candidate_path = shutil.which(
            candidate_name
        )

        if candidate_path is not None:
            detected_path = candidate_path
            break

    tool_records.append({
        "Tool": tool_name,
        "Available": detected_path is not None,
        "Path": detected_path
    })


tool_status = pd.DataFrame(
    tool_records
)

print("本机Git与R环境：")

display(tool_status)


# ------------------------------------------------------------
# 2. 尝试读取版本信息
# ------------------------------------------------------------

version_commands = {
    "Git": [
        "git",
        "--version"
    ],
    "R": [
        "R",
        "--version"
    ],
    "Rscript": [
        "Rscript",
        "--version"
    ]
}

print("\n版本信息：")

for tool_name, command in version_commands.items():

    tool_available = bool(
        tool_status.loc[
            tool_status["Tool"] == tool_name,
            "Available"
        ].iloc[0]
    )

    if not tool_available:

        print(
            f"{tool_name}：未检测到"
        )

        continue

    try:

        completed_process = subprocess.run(
            command,
            capture_output=True,
            text=True,
            errors="ignore",
            timeout=20
        )

        version_text = (
            completed_process.stdout.strip()
            or completed_process.stderr.strip()
        )

        first_line = (
            version_text.splitlines()[0]
            if version_text
            else "未返回版本信息"
        )

        print(
            f"{tool_name}：{first_line}"
        )

    except Exception as error:

        print(
            f"{tool_name}：版本检查失败，{error}"
        )


# ------------------------------------------------------------
# 3. 输出当前Python信息
# ------------------------------------------------------------

print(
    "\n当前操作系统：",
    platform.platform()
)

print(
    "当前Python：",
    sys.executable
)

print(
    "\n第2章第3步完成："
    "Git与R环境检查结束。"
)

本机Git与R环境：


,Tool,Available,Path
0,Git,False,None
1,R,False,None
2,Rscript,False,None



版本信息：
Git：未检测到
R：未检测到
Rscript：未检测到

当前操作系统： Windows-10-10.0.26200-SP0
当前Python： D:\BaiduNetdiskDownload\Anaconda3\envs\geosimcp_repro\python.exe

第2章第3步完成：Git与R环境检查结束。


### 2.3 GCCM官方R代码结构检查

官方GCCM R版本包含三个核心脚本：

- `basic.r`
- `GCCM.r`
- `GCCM4Lattice.r`

同时提供两类示例：

- polygon面数据；
- raster栅格数据。

当前官方案例目录中没有直接提供不规则点数据案例，因此需要进一步检查核心函数的输入形式，判断Seattle住房点数据是否可以直接使用，或者是否需要构造适用于点数据的空间邻接关系。

本节暂不运行GCCM算法，只分析官方R源代码中的：

1. 依赖R包；
2. `source()`调用关系；
3. 核心函数名称；
4. 函数参数接口。

In [7]:
# ============================================================
# 第2章第4步：解析官方GCCM三个核心R脚本
# ============================================================

import re
from pathlib import Path

OFFICIAL_GCCM_ROOT = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "02_gccm"
    / "GCCM_updated 2"
)

OFFICIAL_R_CORE = (
    OFFICIAL_GCCM_ROOT
    / "R"
    / "GCCM"
)

core_r_files = [
    OFFICIAL_R_CORE / "basic.r",
    OFFICIAL_R_CORE / "GCCM.r",
    OFFICIAL_R_CORE / "GCCM4Lattice.r"
]


# ------------------------------------------------------------
# 1. 检查三个文件是否存在
# ------------------------------------------------------------

for file_path in core_r_files:

    print(
        file_path.name,
        "存在：",
        file_path.exists()
    )

    if not file_path.exists():
        raise FileNotFoundError(file_path)


# ------------------------------------------------------------
# 2. 尝试用不同编码读取R源码
# ------------------------------------------------------------

def read_r_file(file_path):

    candidate_encodings = [
        "utf-8",
        "utf-8-sig",
        "gb18030",
        "latin1"
    ]

    for encoding in candidate_encodings:

        try:
            text = file_path.read_text(
                encoding=encoding
            )

            return text, encoding

        except UnicodeDecodeError:
            continue

    raise UnicodeDecodeError(
        "unknown",
        b"",
        0,
        1,
        f"无法读取文件：{file_path}"
    )


# ------------------------------------------------------------
# 3. 提取依赖包、source调用和函数定义
# ------------------------------------------------------------

source_summary = []

for file_path in core_r_files:

    r_text, used_encoding = read_r_file(
        file_path
    )

    lines = r_text.splitlines()

    print("\n" + "=" * 90)
    print("文件：", file_path.name)
    print("编码：", used_encoding)
    print("总行数：", len(lines))
    print("=" * 90)

    package_lines = []

    source_lines = []

    function_lines = []

    for line_number, line in enumerate(
        lines,
        start=1
    ):

        stripped = line.strip()

        # R包
        if re.search(
            r"\b(library|require|requireNamespace)\s*\(",
            stripped
        ):
            package_lines.append(
                (line_number, stripped)
            )

        # source调用
        if re.search(
            r"\bsource\s*\(",
            stripped
        ):
            source_lines.append(
                (line_number, stripped)
            )

        # 函数定义
        if re.search(
            r"<-\s*function\s*\(",
            stripped
        ):
            function_lines.append(
                (line_number, stripped)
            )

    print("\n【依赖包】")

    if package_lines:
        for line_number, line in package_lines:
            print(f"{line_number}: {line}")
    else:
        print("未发现library()/require()调用")

    print("\n【source调用】")

    if source_lines:
        for line_number, line in source_lines:
            print(f"{line_number}: {line}")
    else:
        print("未发现source()调用")

    print("\n【函数定义】")

    if function_lines:
        for line_number, line in function_lines:
            print(f"{line_number}: {line}")
    else:
        print("未发现 '<- function(' 形式的函数定义")

    source_summary.append({
        "File": file_path.name,
        "Encoding": used_encoding,
        "Lines": len(lines),
        "Packages found": len(package_lines),
        "Source calls": len(source_lines),
        "Functions found": len(function_lines)
    })


# ------------------------------------------------------------
# 4. 汇总
# ------------------------------------------------------------

official_r_summary = pd.DataFrame(
    source_summary
)

print("\n官方GCCM核心R脚本汇总：")

display(official_r_summary)

print(
    "\n第2章第4步完成："
    "已提取官方GCCM核心脚本结构。"
)

basic.r 存在： True
GCCM.r 存在： True
GCCM4Lattice.r 存在： True

文件： basic.r
编码： utf-8
总行数： 557

【依赖包】
未发现library()/require()调用

【source调用】
未发现source()调用

【函数定义】
2: confidence<- function(r,n,level=0.05)
24: significance<- function(r,n)
32: expandMatrix<- function(dataMatrix,lagNum)
60: laggedVariable<-function(dataMatrix,lagNum)
115: laggedVariableAs2Dim<-function(dataMatrix,lagNum)
172: distance<-function(emd1,emd2)
189: distanceCon<-function(emd1,emd2)
201: distanceCat<-function(emd1,emd2)
228: distanceCross<-function(trainData,embeddings,neigborNum,targetVar,xName="x",yName="y")
295: distanceCrossGeo<-function(sample,xName="x",yName="y")
315: distanceBtwGeo<-function(test,train, xName="x",yName="y")
332: distanceBtw<-function(predEmd,embeddings)
361: getGridEmbedings<-function(data,nrow,ncol,byrow,atts,lags)
387: getSampleEmbedings<-function(ids,embeddings)
406: normalize <- function(x)
413: dominate<-function(solutions)
453: distanceCrossSign<-function(data,targetVars)
494: #  distance<-f

,File,Encoding,Lines,Packages found,Source calls,Functions found
0,basic.r,utf-8,557,0,0,20
1,GCCM.r,utf-8,326,0,0,6
2,GCCM4Lattice.r,utf-8,322,0,0,7



第2章第4步完成：已提取官方GCCM核心脚本结构。


### 2.4 GCCM核心实现的初步判断

根据官方三个核心R脚本的函数接口，可以初步区分两类实现：

- `GCCM.r` 使用 `xMatrix`、`yMatrix`、`totalRow`、`totalCol` 等规则行列结构，更接近栅格空间数据；
- `GCCM4Lattice.r` 使用 `spNeighbor`、`neighborMatrix` 和 `x_vectors` 等邻接结构，更可能适用于非规则空间单元。

Seattle住房数据属于不规则空间点数据，因此 `GCCM4Lattice.r` 是当前最值得重点检查的官方实现。

不过，仅根据函数名称还不能确认点数据是否可以直接使用。下一步需要检查：

1. 空间滞后如何由邻接矩阵构造；
2. embedding如何生成；
3. GCCMLattice如何执行cross mapping；
4. 官方polygon案例如何构造邻接关系并调用GCCMLattice。

在确认以上实现细节前，不自行定义Seattle点数据的KNN邻接规则。

In [8]:
# ============================================================
# 第2章第5步：提取GCCMLattice核心源码与polygon官方案例
# ============================================================

OFFICIAL_R_DIR = (
    PROJECT_ROOT
    / "GCCP_Seattle"
    / "02_gccm"
    / "GCCM_updated 2"
    / "R"
)

LATTICE_FILE = (
    OFFICIAL_R_DIR
    / "GCCM"
    / "GCCM4Lattice.r"
)

POLYGON_CASE_FILE = (
    OFFICIAL_R_DIR
    / "Cases"
    / "polygon"
    / "CountyPopBatch.r"
)


# ------------------------------------------------------------
# 1. 定义按行号打印源码的函数
# ------------------------------------------------------------

def print_source_range(
    file_path,
    start_line,
    end_line,
    title
):
    text = file_path.read_text(
        encoding="utf-8",
        errors="ignore"
    )

    lines = text.splitlines()

    print("\n" + "=" * 100)
    print(title)
    print("文件：", file_path.name)
    print(
        f"行号：{start_line} - {end_line}"
    )
    print("=" * 100)

    actual_end = min(
        end_line,
        len(lines)
    )

    for line_number in range(
        start_line,
        actual_end + 1
    ):
        print(
            f"{line_number:4d}: "
            f"{lines[line_number - 1]}"
        )


# ------------------------------------------------------------
# 2. 查看空间滞后构造
# ------------------------------------------------------------

print_source_range(
    LATTICE_FILE,
    43,
    101,
    "A. laggedVar4Lattic：空间滞后如何构造"
)


# ------------------------------------------------------------
# 3. 查看embedding构造
# ------------------------------------------------------------

print_source_range(
    LATTICE_FILE,
    102,
    127,
    "B. generateEmbedings：状态空间如何构造"
)


# ------------------------------------------------------------
# 4. 查看GCCMLattice主函数
# ------------------------------------------------------------

print_source_range(
    LATTICE_FILE,
    128,
    204,
    "C. GCCMLattice：核心GCCM流程"
)


# ------------------------------------------------------------
# 5. 查看simplex projection
# ------------------------------------------------------------

print_source_range(
    LATTICE_FILE,
    256,
    321,
    "D. simplex_projection：cross mapping预测过程"
)


# ------------------------------------------------------------
# 6. 查看官方polygon案例
# ------------------------------------------------------------

polygon_text = POLYGON_CASE_FILE.read_text(
    encoding="utf-8",
    errors="ignore"
)

polygon_lines = polygon_text.splitlines()

print("\n" + "=" * 100)
print("E. 官方polygon案例 CountyPopBatch.r")
print("=" * 100)

for line_number, line in enumerate(
    polygon_lines,
    start=1
):
    print(
        f"{line_number:4d}: {line}"
    )


print(
    "\n第2章第5步完成："
    "已提取GCCMLattice核心源码和polygon官方案例。"
)


A. laggedVar4Lattic：空间滞后如何构造
文件： GCCM4Lattice.r
行号：43 - 101
  43: laggedVar4Lattic<- function (spNeighbor,lagNum)
  44: {
  45:   
  46:   lagSpNeighbor<- spNeighbor
  47:   
  48:   if (lagNum<1)
  49:   {
  50:     
  51:     return (NULL)
  52:   }else if(lagNum>1)
  53:   {
  54:     #preSpNeighbor<-spNeighbor
  55:     
  56:     curSpNeighbor<-spNeighbor
  57:     
  58:     for(lag in 1:(lagNum-1) )
  59:     {
  60:       
  61:       preSpNeighbor<-curSpNeighbor
  62:       
  63:       for(i in 1: length(preSpNeighbor))
  64:       {
  65:         curChain <-preSpNeighbor[[i]]
  66:         
  67:         newRings<-curChain
  68:         
  69:         for(neigh in 1: length (curChain))
  70:         {
  71:           nextChainID<-curChain[neigh]
  72:           
  73:           
  74:           if(nextChainID>0)
  75:           {
  76:             nextChain<- spNeighbor[[nextChainID]]
  77:             
  78:             newRings<-unique(c(newRings, nextChain))
  79:       

### 2.5 当前GCCM方法理解

官方polygon版GCCM首先根据空间拓扑关系建立一阶邻接表，然后沿邻接图逐层构造高阶空间滞后。

对于每个空间单元和每一个空间滞后阶数，官方实现使用该阶邻居变量值的平均值构造状态空间维度。随后，在重构状态空间中使用Simplex Projection进行Cross Mapping预测，并以预测值和真实值之间的Pearson相关系数rho衡量Cross Mapping Skill。

Seattle住房属于不规则点数据，不存在polygon数据天然的Queen邻接关系。因此，后续需要根据UTM坐标构造点数据空间邻接图，再沿用官方GCCMLattice算法。

这一邻接图属于从polygon空间单元向point空间单元的适配步骤，需要明确说明并进行邻居参数敏感性分析。

此外，官方polygon案例在GCCM之前分别去除了原因变量和结果变量关于空间坐标的线性趋势。因此Seattle GCCM实验也需要保留这一预处理步骤。

In [9]:
# ============================================================
# 第2章第6步：检查GCCMSingle4Lattice的library抽样过程
# ============================================================

print_source_range(
    LATTICE_FILE,
    1,
    42,
    "GCCMSingle4Lattice：library size与重复抽样机制"
)


GCCMSingle4Lattice：library size与重复抽样机制
文件： GCCM4Lattice.r
行号：1 - 42
   1: GCCMSingle4Lattice<-function(x_vectors,y,lib_indices,lib_size,max_lib_size,possible_lib_indices,pred_indices,b)
   2: {
   3:   n <- NROW(x_vectors) 
   4:    x_xmap_y <- data.frame()
   5:   
   6:    if(lib_size == max_lib_size) # no possible lib variation if using all vectors
   7:   {
   8:     lib_indices <- rep.int(FALSE, times = n)
   9:     lib_indices[possible_lib_indices] <- TRUE
  10:     
  11:     # run cross map and store results
  12:     results <- simplex_projection(x_vectors, y, lib_indices, pred_indices, b)
  13:     x_xmap_y <- rbind(x_xmap_y, data.frame(L = lib_size, rho = results$stats$rho))
  14:   }
  15:   else
  16:   {
  17:     for(start_lib in 1:max_lib_size)
  18:     {
  19:       lib_indices <- rep.int(FALSE, times = n)
  20:       # setup changing library
  21:       if(start_lib+lib_size-1 > max_lib_size) # loop around to beginning of lib indices
  22:       {
  23:         lib_

## 3. Seattle点数据空间邻接结构

官方GCCM的polygon实现通过空间拓扑邻接关系定义一阶空间邻居，并沿邻接图逐层构造高阶空间滞后。

Seattle住房数据属于不规则点数据，不存在polygon数据天然的共享边或共享顶点关系。因此，本实验需要将GCCM的空间邻接思想扩展到点数据。

主实验采用基于UTM坐标构建的Delaunay三角网定义一阶空间邻接关系。

采用Delaunay邻接的原因是：

- 它能够为不规则空间点构造局部拓扑邻接关系；
- 邻接关系是对称的；
- 不需要人为指定固定邻居数量k；
- 与polygon GCCM基于拓扑邻接构建空间滞后的思想较为一致。

在内部参数阶段，仅使用Inner Fit中的1470个住房点构造邻接图，不使用Inner Calibration、Inner Validation、Outer Calibration或Outer Test的空间位置。

后续将另外采用KNN邻接进行敏感性分析，以检查GCCM结果是否过度依赖点邻接定义。

In [11]:
# ============================================================
# 第3章第1步：构造Inner Fit的Delaunay空间邻接图
# ============================================================

from scipy.spatial import Delaunay
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from itertools import combinations


# ------------------------------------------------------------
# 1. 提取Inner Fit的UTM坐标
# ------------------------------------------------------------

inner_coordinates = (
    coordinates_inner_fit[
        ["UTM_X", "UTM_Y"]
    ]
    .to_numpy(dtype=float)
)

n_inner = len(inner_coordinates)


# ------------------------------------------------------------
# 2. 检查完全重复的空间坐标
# ------------------------------------------------------------

coordinate_duplicates = (
    pd.DataFrame(
        inner_coordinates,
        columns=["UTM_X", "UTM_Y"]
    )
    .duplicated()
    .sum()
)

print(
    "Inner Fit完全重复坐标数量：",
    int(coordinate_duplicates)
)

assert coordinate_duplicates == 0


# ------------------------------------------------------------
# 3. 构建Delaunay三角网
# ------------------------------------------------------------

delaunay_inner = Delaunay(
    inner_coordinates
)


# ------------------------------------------------------------
# 4. 从三角形提取无向邻接关系
# ------------------------------------------------------------

inner_neighbor_sets = [
    set()
    for _ in range(n_inner)
]

for simplex in delaunay_inner.simplices:

    for i, j in combinations(
        simplex,
        2
    ):

        i = int(i)
        j = int(j)

        inner_neighbor_sets[i].add(j)
        inner_neighbor_sets[j].add(i)


# ------------------------------------------------------------
# 5. 统计每个住房的一阶邻居数量
# ------------------------------------------------------------

inner_degrees = np.array([
    len(neighbors)
    for neighbors in inner_neighbor_sets
])


degree_summary = pd.DataFrame({
    "Statistic": [
        "Minimum",
        "25%",
        "Median",
        "Mean",
        "75%",
        "Maximum"
    ],
    "Number of first-order neighbors": [
        np.min(inner_degrees),
        np.percentile(inner_degrees, 25),
        np.median(inner_degrees),
        np.mean(inner_degrees),
        np.percentile(inner_degrees, 75),
        np.max(inner_degrees)
    ]
})

print("\nDelaunay一阶邻居数量：")

display(
    degree_summary.round(4)
)


# ------------------------------------------------------------
# 6. 构造稀疏邻接矩阵并检查图是否连通
# ------------------------------------------------------------

row_indices = []
col_indices = []

for i, neighbors in enumerate(
    inner_neighbor_sets
):

    for j in neighbors:

        row_indices.append(i)
        col_indices.append(j)


adjacency_inner = csr_matrix(
    (
        np.ones(
            len(row_indices),
            dtype=int
        ),
        (
            row_indices,
            col_indices
        )
    ),
    shape=(n_inner, n_inner)
)


number_of_components, component_labels = (
    connected_components(
        adjacency_inner,
        directed=False
    )
)

print(
    "空间邻接图连通分量数量：",
    number_of_components
)


# ------------------------------------------------------------
# 7. 计算Delaunay边的实际距离
# ------------------------------------------------------------

edge_distances = []

seen_edges = set()

for i, neighbors in enumerate(
    inner_neighbor_sets
):

    for j in neighbors:

        edge = tuple(
            sorted((i, j))
        )

        if edge in seen_edges:
            continue

        seen_edges.add(edge)

        distance_ij = np.linalg.norm(
            inner_coordinates[i]
            - inner_coordinates[j]
        )

        edge_distances.append(
            distance_ij
        )


edge_distances = np.asarray(
    edge_distances
)


distance_summary = pd.DataFrame({
    "Statistic": [
        "Minimum",
        "25%",
        "Median",
        "Mean",
        "75%",
        "95%",
        "Maximum"
    ],
    "Edge distance (meters)": [
        np.min(edge_distances),
        np.percentile(edge_distances, 25),
        np.median(edge_distances),
        np.mean(edge_distances),
        np.percentile(edge_distances, 75),
        np.percentile(edge_distances, 95),
        np.max(edge_distances)
    ]
})

print("\nDelaunay邻接边距离：")

display(
    distance_summary.round(2)
)


# ------------------------------------------------------------
# 8. 基础检查
# ------------------------------------------------------------

assert len(inner_neighbor_sets) == 1470

assert np.all(
    inner_degrees > 0
)

assert number_of_components == 1


print(
    "\nInner Fit住房数量：",
    n_inner
)

print(
    "Delaunay三角形数量：",
    len(delaunay_inner.simplices)
)

print(
    "无向邻接边数量：",
    len(seen_edges)
)

print(
    "平均一阶邻居数：",
    round(
        float(np.mean(inner_degrees)),
        4
    )
)

print(
    "\n第3章第1步完成："
    "Inner Fit Delaunay空间邻接图构建成功。"
)

Inner Fit完全重复坐标数量： 6


AssertionError: 

In [12]:
# ============================================================
# 第3章第1步A：检查Inner Fit中的重复空间坐标
# ============================================================

inner_fit_check = (
    data.iloc[inner_fit_indices]
    [
        [
            "UTM_X",
            "UTM_Y",
            "bathrooms",
            "sqft_living",
            "sqft_lot",
            "grade",
            "condition",
            "waterfront",
            "view",
            "age",
            "price"
        ]
    ]
    .copy()
)

inner_fit_check["original_index"] = inner_fit_indices


# ------------------------------------------------------------
# 1. 找出所有坐标重复的住房记录
# ------------------------------------------------------------

duplicate_coordinate_mask = (
    inner_fit_check.duplicated(
        subset=["UTM_X", "UTM_Y"],
        keep=False
    )
)

duplicate_coordinate_records = (
    inner_fit_check[
        duplicate_coordinate_mask
    ]
    .sort_values(
        ["UTM_X", "UTM_Y"]
    )
    .copy()
)


# ------------------------------------------------------------
# 2. 按空间位置统计重复组
# ------------------------------------------------------------

duplicate_location_summary = (
    duplicate_coordinate_records
    .groupby(
        ["UTM_X", "UTM_Y"],
        as_index=False
    )
    .agg(
        record_count=(
            "original_index",
            "size"
        ),
        price_min=(
            "price",
            "min"
        ),
        price_max=(
            "price",
            "max"
        ),
        price_mean=(
            "price",
            "mean"
        )
    )
    .sort_values(
        "record_count",
        ascending=False
    )
)


# ------------------------------------------------------------
# 3. 输出总体情况
# ------------------------------------------------------------

number_of_duplicate_records = int(
    duplicate_coordinate_mask.sum()
)

number_of_duplicate_locations = len(
    duplicate_location_summary
)

number_of_unique_locations = (
    inner_fit_check[
        ["UTM_X", "UTM_Y"]
    ]
    .drop_duplicates()
    .shape[0]
)

maximum_records_per_location = (
    int(
        duplicate_location_summary[
            "record_count"
        ].max()
    )
    if number_of_duplicate_locations > 0
    else 1
)


print(
    "Inner Fit住房记录数量：",
    len(inner_fit_check)
)

print(
    "唯一空间位置数量：",
    number_of_unique_locations
)

print(
    "处于重复坐标位置的住房记录数量：",
    number_of_duplicate_records
)

print(
    "重复空间位置数量：",
    number_of_duplicate_locations
)

print(
    "单一位置最多对应住房记录数：",
    maximum_records_per_location
)


# ------------------------------------------------------------
# 4. 显示重复程度最大的前20个位置
# ------------------------------------------------------------

print(
    "\n重复程度最大的空间位置："
)

display(
    duplicate_location_summary
    .head(20)
    .round(4)
)


# ------------------------------------------------------------
# 5. 显示前30条具体重复记录
# 检查它们是否只是坐标相同、住房属性不同
# ------------------------------------------------------------

print(
    "\n部分重复坐标住房记录："
)

display(
    duplicate_coordinate_records
    .head(30)
    .round(4)
)

Inner Fit住房记录数量： 1470
唯一空间位置数量： 1464
处于重复坐标位置的住房记录数量： 12
重复空间位置数量： 6
单一位置最多对应住房记录数： 2

重复程度最大的空间位置：


,UTM_X,UTM_Y,record_count,price_min,price_max,price_mean
0,545340.0259,5.280078e+06,2,45.0,50.0,47.50
1,546671.5445,5.263761e+06,2,26.0,57.5,41.75
2,552831.3122,5.273550e+06,2,40.0,42.0,41.00
3,554396.4532,5.266895e+06,2,36.6,46.5,41.55
4,554797.5374,5.264243e+06,2,16.0,21.0,18.50
5,557536.4732,5.261302e+06,2,20.3,33.9,27.10



部分重复坐标住房记录：


,UTM_X,UTM_Y,bathrooms,sqft_living,sqft_lot,grade,condition,waterfront,view,age,price,original_index
2860,545340.0259,5.280078e+06,1.00,1020,5000,6,4,0,0,109,45.0,2860
1598,545340.0259,5.280078e+06,1.00,1440,7100,7,3,0,0,109,50.0,1598
675,546671.5445,5.263761e+06,1.75,1580,7344,7,5,0,0,104,57.5,675
198,546671.5445,5.263761e+06,1.75,1580,7344,7,5,0,0,104,26.0,198
1534,552831.3122,5.273550e+06,2.50,1170,1811,8,3,0,0,14,40.0,1534
1625,552831.3122,5.273550e+06,1.75,1200,3136,7,3,0,0,111,42.0,1625
2859,554396.4532,5.266895e+06,1.00,930,3348,6,4,0,0,89,46.5,2859
1879,554396.4532,5.266895e+06,1.00,1210,3090,6,3,0,0,89,36.6,1879
1147,554797.5374,5.264243e+06,2.00,1040,4750,6,2,0,0,65,16.0,1147
2035,554797.5374,5.264243e+06,1.00,1450,4750,7,3,0,0,65,21.0,2035


### 3.1 重复空间位置与数据划分检查

Inner Fit中共有1470条住房记录，对应1464个唯一空间位置。

其中仅有6个空间位置出现重复坐标，每个位置均对应2条住房记录。检查发现这些记录的住房属性或房价并不完全相同，因此它们不是简单重复样本，不进行删除。

对于点数据GCCM，后续计划在唯一空间位置上构建Delaunay拓扑邻接关系，同时保留原始住房记录。

在正式构建空间邻接图之前，还需要检查相同空间位置的住房记录是否跨越Train、Calibration和Test集合，以避免同址样本导致潜在的空间信息泄漏。

In [13]:
# ============================================================
# 第3章第1步B：检查重复坐标是否跨数据划分
# ============================================================

# ------------------------------------------------------------
# 1. 给3000条记录标记其所属集合
# ------------------------------------------------------------

split_label = np.full(
    len(data),
    "UNASSIGNED",
    dtype=object
)

split_label[train_indices] = "Outer Train"
split_label[calibration_indices] = "Outer Calibration"
split_label[test_indices] = "Outer Test"


full_split_check = data[
    [
        "UTM_X",
        "UTM_Y",
        "bathrooms",
        "sqft_living",
        "grade",
        "age",
        "price"
    ]
].copy()

full_split_check["split"] = split_label
full_split_check["original_index"] = np.arange(len(data))


# ------------------------------------------------------------
# 2. 找出全数据中所有重复坐标
# ------------------------------------------------------------

duplicate_mask_full = (
    full_split_check.duplicated(
        subset=["UTM_X", "UTM_Y"],
        keep=False
    )
)

duplicate_records_full = (
    full_split_check[
        duplicate_mask_full
    ]
    .sort_values(
        ["UTM_X", "UTM_Y", "split"]
    )
    .copy()
)


# ------------------------------------------------------------
# 3. 对每一个重复位置统计出现在哪些数据集
# ------------------------------------------------------------

duplicate_split_summary = (
    duplicate_records_full
    .groupby(
        ["UTM_X", "UTM_Y"]
    )
    .agg(
        record_count=(
            "original_index",
            "size"
        ),
        split_count=(
            "split",
            "nunique"
        ),
        splits=(
            "split",
            lambda x: " | ".join(
                sorted(set(x))
            )
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# 4. 找出跨集合的同址记录
# ------------------------------------------------------------

cross_split_locations = (
    duplicate_split_summary[
        duplicate_split_summary[
            "split_count"
        ] > 1
    ]
    .copy()
)


print(
    "全数据住房记录数：",
    len(data)
)

print(
    "全数据唯一空间位置数：",
    data[
        ["UTM_X", "UTM_Y"]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "全数据重复空间位置数：",
    len(duplicate_split_summary)
)

print(
    "跨 Outer Train / Calibration / Test 的重复位置数：",
    len(cross_split_locations)
)


print("\n跨集合重复位置：")

display(
    cross_split_locations
)


# ------------------------------------------------------------
# 5. 如果存在跨集合位置，显示具体住房记录
# ------------------------------------------------------------

if len(cross_split_locations) > 0:

    cross_coords = (
        cross_split_locations[
            ["UTM_X", "UTM_Y"]
        ]
    )

    cross_split_records = (
        duplicate_records_full.merge(
            cross_coords,
            on=["UTM_X", "UTM_Y"],
            how="inner"
        )
        .sort_values(
            ["UTM_X", "UTM_Y", "split"]
        )
    )

    print(
        "\n跨集合位置对应的具体住房记录："
    )

    display(
        cross_split_records
    )

else:

    print(
        "\n没有发现同一空间位置跨越"
        "Outer Train / Calibration / Test。"
    )


print(
    "\n第3章第1步B完成："
    "重复坐标与数据划分检查完成。"
)

全数据住房记录数： 3000
全数据唯一空间位置数： 2974
全数据重复空间位置数： 25
跨 Outer Train / Calibration / Test 的重复位置数： 13

跨集合重复位置：


,UTM_X,UTM_Y,record_count,split_count,splits
1,546486.817532,5.239509e+06,2,2,Outer Test | Outer Train
3,546947.326568,5.276256e+06,2,2,Outer Test | Outer Train
4,547964.730411,5.262127e+06,2,2,Outer Calibration | Outer Train
6,548878.209247,5.278906e+06,2,2,Outer Test | Outer Train
8,549068.183282,5.283087e+06,2,2,Outer Test | Outer Train
9,550455.587338,5.261447e+06,2,2,Outer Test | Outer Train
11,552815.112108,5.283609e+06,2,2,Outer Test | Outer Train
15,555647.971331,5.237999e+06,2,2,Outer Test | Outer Train
17,558041.038181,5.286215e+06,2,2,Outer Test | Outer Train
18,561909.226456,5.282142e+06,2,2,Outer Calibration | Outer Train



跨集合位置对应的具体住房记录：


,UTM_X,UTM_Y,bathrooms,sqft_living,grade,age,price,split,original_index
0,546486.817532,5.239509e+06,1.75,1820,8,28,24.7000,Outer Test,96
1,546486.817532,5.239509e+06,2.75,1770,8,36,23.5000,Outer Train,2550
2,546947.326568,5.276256e+06,1.00,980,8,5,44.7000,Outer Test,239
3,546947.326568,5.276256e+06,2.50,1180,8,5,45.5000,Outer Train,153
4,547964.730411,5.262127e+06,2.50,2130,8,1,39.2000,Outer Calibration,1088
5,547964.730411,5.262127e+06,2.50,2130,8,1,38.0000,Outer Train,2335
6,548878.209247,5.278906e+06,3.25,1380,8,7,48.2000,Outer Test,366
7,548878.209247,5.278906e+06,3.25,1380,8,7,47.5000,Outer Train,743
8,549068.183282,5.283087e+06,2.50,1530,8,6,36.0000,Outer Test,2441
9,549068.183282,5.283087e+06,2.50,1530,8,6,37.5000,Outer Train,2768



第3章第1步B完成：重复坐标与数据划分检查完成。


### 3.2 基于空间位置分组的数据划分

进一步检查发现，原始record-level随机划分中存在少量相同UTM坐标跨越Outer Train、Calibration和Test的情况。

这种情况并不直接使用Test标签进行训练，因此不属于传统意义上的标签泄漏。但是，对于依赖空间邻近关系的GeoCP、GeoSIMCP和GCCP，相同空间位置跨集合会产生零地理距离，从而降低空间外推评价的严格性。

因此，主实验改用location-grouped split：

- 将完全相同的`UTM_X + UTM_Y`定义为同一个空间位置组；
- 同一空间位置下的所有住房记录必须进入同一数据子集；
- Outer层仍保持约70% Train、15% Calibration、15% Test；
- Inner层仍仅在Outer Train内部进行约70% Fit、15% Calibration、15% Validation划分；
- 所有基线方法和GCCP均使用同一套固定分组划分。

原record-level split结果保留为初步实验结果，不作为最终主实验划分。

In [14]:
# ============================================================
# 第3章第2步：按空间位置进行Grouped Outer Split
# ============================================================

from sklearn.model_selection import GroupShuffleSplit


# ------------------------------------------------------------
# 1. 每一个唯一UTM位置定义一个group
# ------------------------------------------------------------

location_groups = pd.factorize(
    pd.MultiIndex.from_frame(
        data[["UTM_X", "UTM_Y"]]
    )
)[0]

print(
    "住房记录总数：",
    len(data)
)

print(
    "唯一空间位置数：",
    len(np.unique(location_groups))
)


# ------------------------------------------------------------
# 2. Outer层：
# 70% Train / 30% Temporary
# ------------------------------------------------------------

all_indices = np.arange(
    len(data)
)

outer_split_1 = GroupShuffleSplit(
    n_splits=1,
    train_size=0.70,
    random_state=42
)

train_idx_grouped, temp_idx_grouped = next(
    outer_split_1.split(
        all_indices,
        groups=location_groups
    )
)


# ------------------------------------------------------------
# 3. Temporary再按空间group一分为二
# 得到约15% Calibration / 15% Test
# ------------------------------------------------------------

temp_groups = location_groups[
    temp_idx_grouped
]

outer_split_2 = GroupShuffleSplit(
    n_splits=1,
    train_size=0.50,
    random_state=43
)

cal_rel_idx, test_rel_idx = next(
    outer_split_2.split(
        temp_idx_grouped,
        groups=temp_groups
    )
)

calibration_idx_grouped = (
    temp_idx_grouped[
        cal_rel_idx
    ]
)

test_idx_grouped = (
    temp_idx_grouped[
        test_rel_idx
    ]
)


# ------------------------------------------------------------
# 4. 检查记录数
# ------------------------------------------------------------

print("\nGrouped Outer Split记录数：")

print(
    "Outer Train:",
    len(train_idx_grouped),
    f"({len(train_idx_grouped)/len(data):.4f})"
)

print(
    "Outer Calibration:",
    len(calibration_idx_grouped),
    f"({len(calibration_idx_grouped)/len(data):.4f})"
)

print(
    "Outer Test:",
    len(test_idx_grouped),
    f"({len(test_idx_grouped)/len(data):.4f})"
)


# ------------------------------------------------------------
# 5. 检查唯一空间位置数量
# ------------------------------------------------------------

train_groups = set(
    location_groups[
        train_idx_grouped
    ]
)

cal_groups = set(
    location_groups[
        calibration_idx_grouped
    ]
)

test_groups = set(
    location_groups[
        test_idx_grouped
    ]
)

print("\n唯一空间位置数量：")

print(
    "Outer Train:",
    len(train_groups)
)

print(
    "Outer Calibration:",
    len(cal_groups)
)

print(
    "Outer Test:",
    len(test_groups)
)


# ------------------------------------------------------------
# 6. 最重要的检查：
# 不同集合之间不能共享空间位置
# ------------------------------------------------------------

train_cal_overlap = (
    train_groups & cal_groups
)

train_test_overlap = (
    train_groups & test_groups
)

cal_test_overlap = (
    cal_groups & test_groups
)

print("\n空间位置交叉检查：")

print(
    "Train ∩ Calibration:",
    len(train_cal_overlap)
)

print(
    "Train ∩ Test:",
    len(train_test_overlap)
)

print(
    "Calibration ∩ Test:",
    len(cal_test_overlap)
)

assert len(train_cal_overlap) == 0
assert len(train_test_overlap) == 0
assert len(cal_test_overlap) == 0


# ------------------------------------------------------------
# 7. 查看price分布
# ------------------------------------------------------------

grouped_outer_summary = pd.DataFrame({
    "Split": [
        "Outer Train",
        "Outer Calibration",
        "Outer Test"
    ],

    "N": [
        len(train_idx_grouped),
        len(calibration_idx_grouped),
        len(test_idx_grouped)
    ],

    "Mean price": [
        data.iloc[
            train_idx_grouped
        ]["price"].mean(),

        data.iloc[
            calibration_idx_grouped
        ]["price"].mean(),

        data.iloc[
            test_idx_grouped
        ]["price"].mean()
    ],

    "Median price": [
        data.iloc[
            train_idx_grouped
        ]["price"].median(),

        data.iloc[
            calibration_idx_grouped
        ]["price"].median(),

        data.iloc[
            test_idx_grouped
        ]["price"].median()
    ]
})

print(
    "\nGrouped Outer Split价格分布："
)

display(
    grouped_outer_summary.round(4)
)


print(
    "\n第3章第2步完成："
    "Grouped Outer Split构建成功。"
)

住房记录总数： 3000
唯一空间位置数： 2974

Grouped Outer Split记录数：
Outer Train: 2102 (0.7007)
Outer Calibration: 450 (0.1500)
Outer Test: 448 (0.1493)

唯一空间位置数量：
Outer Train: 2081
Outer Calibration: 446
Outer Test: 447

空间位置交叉检查：
Train ∩ Calibration: 0
Train ∩ Test: 0
Calibration ∩ Test: 0

Grouped Outer Split价格分布：


,Split,N,Mean price,Median price
0,Outer Train,2102,52.8372,43.75
1,Outer Calibration,450,51.8649,43.10
2,Outer Test,448,54.1627,43.55



第3章第2步完成：Grouped Outer Split构建成功。


## 3.2 Baseline实验前置审计

在进行Seattle GCCM点数据适配时发现，原始record-level随机划分中存在少量完全相同空间坐标跨越Outer Train、Calibration和Test的现象。

因此，在决定是否调整主实验数据划分之前，对当前Baseline实验协议进行一次完整审计。

本次审计不修改任何现有数据划分，也不重新训练模型，主要检查：

- 完全相同空间位置的跨集合情况；
- 近距离空间样本的跨集合情况；
- 相同预测变量组合是否跨集合；
- Train、Calibration和Test的目标变量分布；
- Test样本与Train样本之间的最近空间距离。

审计完成后，再结合GeoSIMCP作者原始实验代码决定是否保留当前record-level split。

In [15]:
# ============================================================
# Baseline Preflight Audit
# 不修改任何已有split
# ============================================================

import numpy as np
import pandas as pd

from sklearn.neighbors import NearestNeighbors


# ------------------------------------------------------------
# 1. 标记当前Outer Split
# ------------------------------------------------------------

outer_split_labels = np.full(
    len(data),
    "UNASSIGNED",
    dtype=object
)

outer_split_labels[train_indices] = "Outer Train"
outer_split_labels[calibration_indices] = "Outer Calibration"
outer_split_labels[test_indices] = "Outer Test"

assert np.all(
    outer_split_labels != "UNASSIGNED"
)


# ------------------------------------------------------------
# 2. 当前模型使用的预测变量
# ------------------------------------------------------------

nonspatial_features = [
    "bathrooms",
    "sqft_living",
    "sqft_lot",
    "grade",
    "condition",
    "waterfront",
    "view",
    "age"
]

model_features = (
    nonspatial_features
    + ["UTM_X", "UTM_Y"]
)


# ------------------------------------------------------------
# 3. 通用函数：
# 检查相同变量组合是否跨Outer Split
# ------------------------------------------------------------

def find_cross_split_groups(
    dataframe,
    columns,
    labels
):

    temp = dataframe[
        columns
    ].copy()

    temp["_split"] = labels

    grouped = (
        temp
        .groupby(
            columns,
            dropna=False
        )
        .agg(
            record_count=(
                "_split",
                "size"
            ),
            split_count=(
                "_split",
                "nunique"
            ),
            splits=(
                "_split",
                lambda x:
                " | ".join(
                    sorted(set(x))
                )
            )
        )
        .reset_index()
    )

    cross_split = (
        grouped[
            grouped[
                "split_count"
            ] > 1
        ]
        .copy()
    )

    return cross_split


# ------------------------------------------------------------
# 4. 完全相同坐标跨集合
# ------------------------------------------------------------

cross_same_coordinate = (
    find_cross_split_groups(
        data,
        ["UTM_X", "UTM_Y"],
        outer_split_labels
    )
)

print(
    "完全相同坐标跨Outer Split的位置数：",
    len(cross_same_coordinate)
)


# ------------------------------------------------------------
# 5. 完全相同8个非空间属性是否跨集合
# ------------------------------------------------------------

cross_same_attributes = (
    find_cross_split_groups(
        data,
        nonspatial_features,
        outer_split_labels
    )
)

print(
    "完全相同8个非空间属性且跨Outer Split的组合数：",
    len(cross_same_attributes)
)


# ------------------------------------------------------------
# 6. 完全相同10个模型输入变量是否跨集合
#    即：属性 + 完全相同空间位置
# ------------------------------------------------------------

cross_same_model_features = (
    find_cross_split_groups(
        data,
        model_features,
        outer_split_labels
    )
)

print(
    "完全相同10个模型输入变量且跨Outer Split的组合数：",
    len(cross_same_model_features)
)


# ------------------------------------------------------------
# 7. Test / Calibration 到Train的最近空间距离
#    UTM单位 = 米
# ------------------------------------------------------------

coordinates_all = (
    data[
        ["UTM_X", "UTM_Y"]
    ]
    .to_numpy(dtype=float)
)

train_coordinates = (
    coordinates_all[
        train_indices
    ]
)

calibration_coordinates = (
    coordinates_all[
        calibration_indices
    ]
)

test_coordinates = (
    coordinates_all[
        test_indices
    ]
)


nn_train = NearestNeighbors(
    n_neighbors=1,
    metric="euclidean"
)

nn_train.fit(
    train_coordinates
)


cal_distances, _ = (
    nn_train.kneighbors(
        calibration_coordinates
    )
)

test_distances, _ = (
    nn_train.kneighbors(
        test_coordinates
    )
)

cal_distances = (
    cal_distances[:, 0]
)

test_distances = (
    test_distances[:, 0]
)


def distance_summary(
    distances,
    split_name
):

    return {
        "Split": split_name,

        "Minimum (m)":
            np.min(distances),

        "25% (m)":
            np.percentile(
                distances,
                25
            ),

        "Median (m)":
            np.median(distances),

        "Mean (m)":
            np.mean(distances),

        "75% (m)":
            np.percentile(
                distances,
                75
            ),

        "95% (m)":
            np.percentile(
                distances,
                95
            ),

        "Maximum (m)":
            np.max(distances),

        "N distance = 0":
            np.sum(
                distances == 0
            ),

        "N <= 10 m":
            np.sum(
                distances <= 10
            ),

        "N <= 50 m":
            np.sum(
                distances <= 50
            ),

        "N <= 100 m":
            np.sum(
                distances <= 100
            )
    }


nearest_distance_summary = pd.DataFrame([
    distance_summary(
        cal_distances,
        "Outer Calibration → Train"
    ),

    distance_summary(
        test_distances,
        "Outer Test → Train"
    )
])


print(
    "\nCalibration/Test到Train的最近空间距离："
)

display(
    nearest_distance_summary.round(2)
)


# ------------------------------------------------------------
# 8. 三个集合的price完整分布
# ------------------------------------------------------------

price_distribution = []

for split_name, indices in [
    (
        "Outer Train",
        train_indices
    ),
    (
        "Outer Calibration",
        calibration_indices
    ),
    (
        "Outer Test",
        test_indices
    )
]:

    values = (
        data.iloc[
            indices
        ]["price"]
    )

    price_distribution.append({
        "Split": split_name,
        "N": len(values),
        "Mean": values.mean(),
        "Std": values.std(),
        "Min": values.min(),
        "25%": values.quantile(0.25),
        "Median": values.median(),
        "75%": values.quantile(0.75),
        "95%": values.quantile(0.95),
        "Max": values.max()
    })


price_distribution = (
    pd.DataFrame(
        price_distribution
    )
)

print(
    "\nOuter Split价格分布："
)

display(
    price_distribution.round(4)
)


# ------------------------------------------------------------
# 9. 显示真正值得重点看的：
# 完全相同10维模型输入却跨集合的记录
# ------------------------------------------------------------

if len(
    cross_same_model_features
) > 0:

    print(
        "\n完全相同模型输入、"
        "但跨Outer Split的组合："
    )

    display(
        cross_same_model_features
        .head(30)
    )

else:

    print(
        "\n没有发现完全相同的10维模型输入"
        "跨Outer Split。"
    )


print(
    "\nBaseline Preflight Audit完成。"
)

完全相同坐标跨Outer Split的位置数： 13
完全相同8个非空间属性且跨Outer Split的组合数： 4
完全相同10个模型输入变量且跨Outer Split的组合数： 4

Calibration/Test到Train的最近空间距离：


,Split,Minimum (m),25% (m),Median (m),Mean (m),75% (m),95% (m),Maximum (m),N distance = 0,N <= 10 m,N <= 50 m,N <= 100 m
0,Outer Calibration → Train,0.0,143.56,224.72,274.43,345.84,641.70,1968.57,4,4,26,79
1,Outer Test → Train,0.0,134.13,235.16,294.49,376.75,704.69,2859.78,9,9,26,80



Outer Split价格分布：


,Split,N,Mean,Std,Min,25%,Median,75%,95%,Max
0,Outer Train,2100,53.1103,37.4867,8.2,31.50,43.42,60.5031,119.525,356.7
1,Outer Calibration,450,53.5939,35.7738,10.0,31.50,45.00,63.0000,120.000,363.5
2,Outer Test,450,51.1534,31.5999,11.5,31.05,43.25,61.9462,99.365,254.6



完全相同模型输入、但跨Outer Split的组合：


,bathrooms,sqft_living,sqft_lot,grade,condition,waterfront,view,age,UTM_X,UTM_Y,record_count,split_count,splits
276,1.0,1080,6250,5,2,0,0,65,550455.587338,5.261447e+06,2,2,Outer Test | Outer Train
1518,2.0,1900,8240,7,2,0,0,51,552815.112108,5.283609e+06,2,2,Outer Test | Outer Train
1967,2.5,1530,1131,8,3,0,0,6,549068.183282,5.283087e+06,2,2,Outer Test | Outer Train
2075,2.5,1940,3211,8,3,0,0,6,568225.453861,5.268286e+06,2,2,Outer Test | Outer Train



Baseline Preflight Audit完成。


In [16]:
# ============================================================
# Baseline Audit 2：
# 检查完全相同模型输入但跨Outer Split的记录
# 同时检查Inner Split是否也存在同址重叠
# ============================================================

# ------------------------------------------------------------
# 1. 找出10个模型输入完全相同、但跨Outer Split的具体记录
# ------------------------------------------------------------

audit_exact = data[
    model_features + ["price"]
].copy()

audit_exact["outer_split"] = outer_split_labels
audit_exact["original_index"] = np.arange(len(data))


# 找出所有10维模型输入重复记录
exact_duplicate_mask = (
    audit_exact.duplicated(
        subset=model_features,
        keep=False
    )
)

exact_duplicate_records = (
    audit_exact[
        exact_duplicate_mask
    ]
    .copy()
)


# 只保留真正跨Outer Split的组
exact_group_info = (
    exact_duplicate_records
    .groupby(
        model_features,
        dropna=False
    )
    .agg(
        record_count=("original_index", "size"),
        split_count=("outer_split", "nunique"),
        price_min=("price", "min"),
        price_max=("price", "max"),
        price_mean=("price", "mean")
    )
    .reset_index()
)

exact_cross_groups = (
    exact_group_info[
        exact_group_info["split_count"] > 1
    ]
    .copy()
)


exact_cross_records = (
    exact_duplicate_records.merge(
        exact_cross_groups[
            model_features
        ],
        on=model_features,
        how="inner"
    )
    .sort_values(
        model_features
    )
)


print(
    "10维模型输入完全相同且跨Outer Split的组数：",
    len(exact_cross_groups)
)

print(
    "涉及的住房记录数量：",
    len(exact_cross_records)
)

print("\n具体记录：")

display(
    exact_cross_records[
        [
            "original_index",
            "outer_split",
            "price",
            "bathrooms",
            "sqft_living",
            "sqft_lot",
            "grade",
            "condition",
            "waterfront",
            "view",
            "age",
            "UTM_X",
            "UTM_Y"
        ]
    ]
)


print("\n每组价格差异：")

display(
    exact_cross_groups[
        model_features
        + [
            "record_count",
            "split_count",
            "price_min",
            "price_max",
            "price_mean"
        ]
    ]
)


# ============================================================
# 2. 检查Inner Split
# ============================================================

inner_labels = np.full(
    len(data),
    "NOT_IN_INNER",
    dtype=object
)

inner_labels[inner_fit_indices] = "Inner Fit"
inner_labels[inner_calibration_indices] = "Inner Calibration"
inner_labels[inner_validation_indices] = "Inner Validation"


inner_outer_train_mask = (
    inner_labels != "NOT_IN_INNER"
)

inner_data = (
    data.loc[
        inner_outer_train_mask,
        model_features + ["price"]
    ]
    .copy()
)

inner_data["inner_split"] = (
    inner_labels[
        inner_outer_train_mask
    ]
)

inner_data["original_index"] = (
    np.arange(len(data))[
        inner_outer_train_mask
    ]
)


# ------------------------------------------------------------
# 3. Inner层相同坐标跨集合
# ------------------------------------------------------------

inner_coord_groups = (
    inner_data
    .groupby(
        ["UTM_X", "UTM_Y"],
        dropna=False
    )
    .agg(
        record_count=("original_index", "size"),
        split_count=("inner_split", "nunique"),
        splits=(
            "inner_split",
            lambda x:
            " | ".join(sorted(set(x)))
        )
    )
    .reset_index()
)

inner_cross_coordinates = (
    inner_coord_groups[
        inner_coord_groups["split_count"] > 1
    ]
)


# ------------------------------------------------------------
# 4. Inner层10维模型输入完全相同且跨集合
# ------------------------------------------------------------

inner_feature_groups = (
    inner_data
    .groupby(
        model_features,
        dropna=False
    )
    .agg(
        record_count=("original_index", "size"),
        split_count=("inner_split", "nunique"),
        splits=(
            "inner_split",
            lambda x:
            " | ".join(sorted(set(x)))
        ),
        price_min=("price", "min"),
        price_max=("price", "max")
    )
    .reset_index()
)

inner_cross_features = (
    inner_feature_groups[
        inner_feature_groups["split_count"] > 1
    ]
)


print("\n" + "=" * 80)

print(
    "Inner层相同坐标跨 Fit / Calibration / Validation 的位置数：",
    len(inner_cross_coordinates)
)

print(
    "Inner层完全相同10维模型输入跨集合的组数：",
    len(inner_cross_features)
)


if len(inner_cross_features) > 0:

    print("\nInner层完全相同模型输入的跨集合组：")

    display(
        inner_cross_features
    )


print(
    "\nBaseline Audit 2完成。"
)

10维模型输入完全相同且跨Outer Split的组数： 4
涉及的住房记录数量： 8

具体记录：


,original_index,outer_split,price,bathrooms,sqft_living,sqft_lot,grade,condition,waterfront,view,age,UTM_X,UTM_Y
2,2235,Outer Train,11.500,1.0,1080,6250,5,2,0,0,65,550455.587338,5.261447e+06
7,2883,Outer Test,15.700,1.0,1080,6250,5,2,0,0,65,550455.587338,5.261447e+06
0,547,Outer Test,30.500,2.0,1900,8240,7,2,0,0,51,552815.112108,5.283609e+06
5,2653,Outer Train,64.995,2.0,1900,8240,7,2,0,0,51,552815.112108,5.283609e+06
3,2441,Outer Test,36.000,2.5,1530,1131,8,3,0,0,6,549068.183282,5.283087e+06
6,2768,Outer Train,37.500,2.5,1530,1131,8,3,0,0,6,549068.183282,5.283087e+06
1,1372,Outer Train,55.500,2.5,1940,3211,8,3,0,0,6,568225.453861,5.268286e+06
4,2525,Outer Test,55.500,2.5,1940,3211,8,3,0,0,6,568225.453861,5.268286e+06



每组价格差异：


,bathrooms,sqft_living,sqft_lot,grade,condition,waterfront,view,age,UTM_X,UTM_Y,record_count,split_count,price_min,price_max,price_mean
0,1.0,1080,6250,5,2,0,0,65,550455.587338,5.261447e+06,2,2,11.5,15.700,13.6000
2,2.0,1900,8240,7,2,0,0,51,552815.112108,5.283609e+06,2,2,30.5,64.995,47.7475
3,2.5,1530,1131,8,3,0,0,6,549068.183282,5.283087e+06,2,2,36.0,37.500,36.7500
4,2.5,1940,3211,8,3,0,0,6,568225.453861,5.268286e+06,2,2,55.5,55.500,55.5000



Inner层相同坐标跨 Fit / Calibration / Validation 的位置数： 6
Inner层完全相同10维模型输入跨集合的组数： 0

Baseline Audit 2完成。


## 4. GeoSIMCP作者实现协议审计

在继续GCCM和GCCP之前，对GeoSIMCP作者原始实现进行一次最终核查。

本节主要确认：

1. Seattle数据在作者实验中如何划分；
2. 预测目标使用`log_price`还是还原后的`price`；
3. XGBoost模型使用哪些预测变量；
4. GeoSIMCP特征距离使用哪些变量；
5. `standardize_weights`如何实现；
6. Euclidean与MND距离的具体计算形式；
7. weighted quantile如何计算；
8. bandwidth和lambda在作者代码中的作用。

完成本节后冻结Seattle主实验协议，后续GCCM和GCCP不再修改Baseline数据划分和评价规则。

In [17]:
# ============================================================
# 第4章第1步：审查GeoSIMCP作者源码与Seattle实验代码
# ============================================================

import json
import re
from pathlib import Path


# ------------------------------------------------------------
# 1. 作者代码路径
# ------------------------------------------------------------

AUTHOR_ROOT = (
    PROJECT_ROOT
    / "Codes_Manuscript_GeoSIMCP"
    / "Codes_GeoSIMCP"
)

GEOCONFORMAL_DIR = (
    AUTHOR_ROOT
    / "GeoConformal"
)


files_to_check = [
    GEOCONFORMAL_DIR / "GeoConformalSpatialRegression.py",
    GEOCONFORMAL_DIR / "GeoSIMCP.py",
    GEOCONFORMAL_DIR / "GeoSIMCPzhao.py"
]


# ------------------------------------------------------------
# 2. 搜索Python源码中的关键实现
# ------------------------------------------------------------

patterns = [
    "standardize",
    "StandardScaler",
    "_weighted_quantile",
    "quantile",
    "lambda",
    "bandwidth",
    "X_calib_weight",
    "X_test_weight",
    "d_geo",
    "d_feat",
    "d_joint",
    "ranges",
    "np.ptp"
]


for file_path in files_to_check:

    print("\n" + "=" * 100)
    print("文件：", file_path)
    print("存在：", file_path.exists())
    print("=" * 100)

    if not file_path.exists():
        continue

    text = file_path.read_text(
        encoding="utf-8",
        errors="ignore"
    )

    lines = text.splitlines()

    for line_no, line in enumerate(
        lines,
        start=1
    ):

        if any(
            pattern.lower() in line.lower()
            for pattern in patterns
        ):

            start = max(
                0,
                line_no - 3
            )

            end = min(
                len(lines),
                line_no + 3
            )

            print(
                f"\n--- around line {line_no} ---"
            )

            for j in range(start, end):

                print(
                    f"{j+1:4d}: {lines[j]}"
                )


# ------------------------------------------------------------
# 3. 搜索作者Notebook中的实验协议
# ------------------------------------------------------------

author_notebooks = list(
    AUTHOR_ROOT.rglob("*.ipynb")
)

print(
    "\n找到作者Notebook数量：",
    len(author_notebooks)
)


notebook_keywords = [
    "seattle",
    "log_price",
    "price",
    "train_test_split",
    "random_state",
    "XGBRegressor",
    "UTM_X",
    "UTM_Y",
    "standardize_weights",
    "lambda_weight",
    "bandwidth"
]


for notebook_path in author_notebooks:

    try:

        notebook = json.loads(
            notebook_path.read_text(
                encoding="utf-8",
                errors="ignore"
            )
        )

    except Exception:
        continue


    matched_cells = []

    for cell_idx, cell in enumerate(
        notebook.get("cells", [])
    ):

        source = "".join(
            cell.get("source", [])
        )

        if any(
            keyword.lower()
            in source.lower()
            for keyword
            in notebook_keywords
        ):

            matched_cells.append(
                (
                    cell_idx,
                    source
                )
            )


    if matched_cells:

        print(
            "\n" + "#" * 100
        )

        print(
            "Notebook：",
            notebook_path.name
        )

        print(
            "路径：",
            notebook_path
        )

        print(
            "#" * 100
        )


        for cell_idx, source in matched_cells:

            print(
                f"\n--- Cell {cell_idx} ---"
            )

            print(
                source[:5000]
            )


print(
    "\n第4章第1步完成："
    "GeoSIMCP作者实现协议审计信息已提取。"
)


文件： E:\GeoSIMCP_Reproduce\geoconformal-main\Codes_Manuscript_GeoSIMCP\Codes_GeoSIMCP\GeoConformal\GeoConformalSpatialRegression.py
存在： True

--- around line 10 ---
   8: class GeoConformalSpatialRegression:
   9:     def __init__(self, predict_f: Callable, nonconformity_score_f: Callable = None,
  10:                  miscoverage_level: float = 0.1, bandwidth: float = None,
  11:                  coord_calib: np.ndarray = None, coord_test: np.ndarray = None,
  12:                  X_calib: np.ndarray = None, y_calib: np.ndarray = None,
  13:                  X_test: np.ndarray = None, y_test: np.ndarray = None):

--- around line 20 ---
  18:         nonconformity_score_f: a measure of how well a new data point conforms to a model trained on a given dataset
  19:         miscoverage_level: the percentage of data points not in the confidence interval
  20:         bandwidth: the bandwidth of the Gaussian kernel
  21:         test_data: data points used for measuring uncertainty
  22:   

In [18]:
# Chapter 3 pre-check: inspect current Seattle GCCM workspace

names_to_check = [
    name for name in globals()
    if any(key in name.lower() for key in [
        "inner", "train", "cal", "test",
        "feature", "causal", "coord", "utm",
        "price", "idx"
    ])
]

print("Relevant variables currently in memory:")
for name in sorted(names_to_check):
    obj = globals()[name]
    try:
        print(f"{name:35s}  type={type(obj).__name__:15s}  shape={getattr(obj, 'shape', None)}")
    except Exception:
        print(name)

Relevant variables currently in memory:
ASpatial_FEATURES                    type=list             shape=None
INNER_SPLIT_PATH                     type=WindowsPath      shape=None
SPATIAL_COORDINATES                  type=list             shape=None
X_causal_all                         type=DataFrame        shape=(3000, 8)
X_causal_inner_fit                   type=DataFrame        shape=(1470, 8)
X_causal_outer_train                 type=DataFrame        shape=(2100, 8)
cal_distances                        type=ndarray          shape=(450,)
cal_groups                           type=set              shape=None
cal_rel_idx                          type=ndarray          shape=(450,)
cal_test_overlap                     type=set              shape=None
calibration_coordinates              type=ndarray          shape=(450, 2)
calibration_idx_grouped              type=ndarray          shape=(450,)
calibration_indices                  type=ndarray          shape=(450,)
calibration_set        

## Chapter 3 — Point-based spatial graph for Seattle GCCM

The official GCCM polygon example defines spatial neighborhoods through polygon adjacency.
Seattle housing data consist of point observations, so a point-based spatial graph is required.

For the main Seattle GCCM experiment:

1. Causal structure is estimated using **Inner Fit only**.
2. Records sharing exactly the same UTM coordinate are aggregated **within Inner Fit only**.
3. The aggregation is performed by spatial site:
   - coordinates remain unchanged;
   - housing attributes are averaged within each site;
   - house price is averaged within each site.
4. A **Delaunay triangulation** is constructed over the unique Inner-Fit sites.
5. Two sites are treated as neighbors if they share a Delaunay triangle edge.

This changes only the spatial adjacency representation required to adapt the official polygon GCCM implementation to point data. Calibration, validation, and test records are not used in causal-structure estimation.

In [19]:
# Chapter 3.1 — Verify alignment of the Inner-Fit causal data

print("X_causal_inner_fit:", X_causal_inner_fit.shape)
print("coordinates_inner_fit:", coordinates_inner_fit.shape)
print("y_inner_fit:", y_inner_fit.shape)

print("\nCausal features:")
print(X_causal_inner_fit.columns.tolist())

print("\nCoordinate columns:")
print(coordinates_inner_fit.columns.tolist())

assert len(X_causal_inner_fit) == len(coordinates_inner_fit) == len(y_inner_fit)

assert X_causal_inner_fit.index.equals(coordinates_inner_fit.index), \
    "X_causal_inner_fit and coordinates_inner_fit indices are not aligned."

assert X_causal_inner_fit.index.equals(y_inner_fit.index), \
    "X_causal_inner_fit and y_inner_fit indices are not aligned."

print("\n✓ Inner-Fit feature / coordinate / target indices are perfectly aligned.")

X_causal_inner_fit: (1470, 8)
coordinates_inner_fit: (1470, 2)
y_inner_fit: (1470,)

Causal features:
['bathrooms', 'sqft_living', 'sqft_lot', 'grade', 'condition', 'waterfront', 'view', 'age']

Coordinate columns:
['UTM_X', 'UTM_Y']

✓ Inner-Fit feature / coordinate / target indices are perfectly aligned.


In [20]:
# Chapter 3.2 — Aggregate co-located Inner-Fit records to unique spatial sites

import numpy as np
import pandas as pd

coord_cols = coordinates_inner_fit.columns.tolist()
causal_feature_cols = X_causal_inner_fit.columns.tolist()

assert len(coord_cols) == 2, "Expected exactly two spatial coordinate columns."

# Assemble the causal-estimation table.
# Keep the current row order because the official GCCM implementation
# later uses row ordering when constructing cyclic libraries.
gccm_inner_records = pd.concat(
    [
        coordinates_inner_fit[coord_cols],
        X_causal_inner_fit[causal_feature_cols],
        y_inner_fit.rename("price")
    ],
    axis=1
).copy()

print("Record-level Inner Fit:", gccm_inner_records.shape)

# Preserve first-occurrence spatial-site order with sort=False.
gccm_inner_sites = (
    gccm_inner_records
    .groupby(coord_cols, sort=False, as_index=False)
    .agg({
        **{col: "mean" for col in causal_feature_cols},
        "price": "mean"
    })
)

print("Unique spatial sites:", gccm_inner_sites.shape)
print("Number of records aggregated:",
      len(gccm_inner_records) - len(gccm_inner_sites))

assert len(gccm_inner_records) == 1470
assert len(gccm_inner_sites) == 1464

print("\n✓ Inner Fit: 1470 records -> 1464 unique spatial sites.")

Record-level Inner Fit: (1470, 11)
Unique spatial sites: (1464, 11)
Number of records aggregated: 6

✓ Inner Fit: 1470 records -> 1464 unique spatial sites.


In [21]:
# Chapter 3.3 — Validate the site-level causal table

site_coords = gccm_inner_sites[coord_cols].to_numpy(dtype=float)

n_duplicate_sites_after = (
    gccm_inner_sites
    .duplicated(subset=coord_cols, keep=False)
    .sum()
)

print("Site coordinate matrix:", site_coords.shape)
print("Duplicate site rows after aggregation:", n_duplicate_sites_after)

print("\nMissing values:")
print(gccm_inner_sites[causal_feature_cols + ["price"]].isna().sum())

assert n_duplicate_sites_after == 0
assert np.isfinite(site_coords).all()
assert np.isfinite(
    gccm_inner_sites[causal_feature_cols + ["price"]].to_numpy(dtype=float)
).all()

print("\n✓ Site-level GCCM table is finite and has unique coordinates.")

Site coordinate matrix: (1464, 2)
Duplicate site rows after aggregation: 0

Missing values:
bathrooms      0
sqft_living    0
sqft_lot       0
grade          0
condition      0
waterfront     0
view           0
age            0
price          0
dtype: int64

✓ Site-level GCCM table is finite and has unique coordinates.


In [22]:
# Chapter 3.4 — Construct the Delaunay spatial adjacency graph

from scipy.spatial import Delaunay

tri = Delaunay(site_coords)

n_sites = len(site_coords)

# Python-side 0-based adjacency sets
neighbor_sets = [set() for _ in range(n_sites)]

# Every triangle contributes three undirected edges.
for simplex in tri.simplices:
    a, b, c = map(int, simplex)

    neighbor_sets[a].update([b, c])
    neighbor_sets[b].update([a, c])
    neighbor_sets[c].update([a, b])

# Convert to sorted arrays.
delaunay_neighbors_0based = [
    np.array(sorted(neigh), dtype=int)
    for neigh in neighbor_sets
]

degrees = np.array(
    [len(neigh) for neigh in delaunay_neighbors_0based],
    dtype=int
)

print("Unique sites:", n_sites)
print("Delaunay triangles:", tri.simplices.shape[0])
print("Undirected edges:", degrees.sum() // 2)

print("\nNeighbor degree summary:")
print(pd.Series(degrees).describe())

print("\nSites with zero neighbors:", np.sum(degrees == 0))

assert n_sites == 1464
assert np.all(degrees > 0)

# Symmetry check
for i, neighs in enumerate(delaunay_neighbors_0based):
    for j in neighs:
        assert i in neighbor_sets[j]

print("\n✓ Delaunay graph constructed successfully.")
print("✓ All sites have at least one neighbor.")
print("✓ Adjacency is symmetric.")

Unique sites: 1464
Delaunay triangles: 2913
Undirected edges: 4376

Neighbor degree summary:
count    1464.000000
mean        5.978142
std         1.497617
min         3.000000
25%         5.000000
50%         6.000000
75%         7.000000
max        13.000000
dtype: float64

Sites with zero neighbors: 0

✓ Delaunay graph constructed successfully.
✓ All sites have at least one neighbor.
✓ Adjacency is symmetric.


In [23]:
# Chapter 3.5 — Inspect several Delaunay neighborhoods

for i in range(5):
    print(
        f"Site {i}: degree={degrees[i]:2d}, "
        f"neighbors={delaunay_neighbors_0based[i].tolist()}"
    )

Site 0: degree= 5, neighbors=[336, 998, 1200, 1230, 1277]
Site 1: degree= 5, neighbors=[19, 229, 1067, 1097, 1173]
Site 2: degree= 5, neighbors=[29, 240, 295, 1060, 1098]
Site 3: degree= 6, neighbors=[90, 157, 323, 533, 794, 1329]
Site 4: degree= 6, neighbors=[231, 344, 456, 944, 1191, 1457]


### 3.6 Spatial-ring audit for E = 3

The official GCCM lattice implementation reconstructs the spatial state space using successive graph-distance rings.

For a focal site \(i\):

- Ring 1 contains directly adjacent sites;
- Ring 2 contains sites reached in exactly two graph steps, excluding Ring 1 and the focal site;
- Ring 3 contains sites reached in exactly three graph steps, excluding all previously reached sites.

Before running GCCM, we verify that the Delaunay graph provides sufficiently populated spatial rings for the selected embedding dimension \(E=3\).

In [24]:
# Chapter 3.6 — Audit graph-distance rings for E = 3

import numpy as np
import pandas as pd

def graph_ring(neighbor_sets, focal, lag):
    """
    Return nodes whose shortest-path graph distance from focal
    is exactly `lag`.

    For an undirected graph, this is equivalent to the spatial-ring
    construction used by the official GCCM laggedVar4Lattic() logic.
    """
    visited = {focal}
    frontier = {focal}

    for _ in range(lag):
        next_frontier = set()

        for node in frontier:
            next_frontier.update(neighbor_sets[node])

        next_frontier -= visited
        visited.update(next_frontier)
        frontier = next_frontier

    return frontier


rings = {}

for lag in [1, 2, 3]:
    rings[lag] = [
        graph_ring(neighbor_sets, i, lag)
        for i in range(n_sites)
    ]

    sizes = np.array([len(r) for r in rings[lag]])

    print(f"\n========== Ring {lag} ==========")
    print(pd.Series(sizes).describe())
    print("Empty rings:", np.sum(sizes == 0))
    print("Minimum size:", sizes.min())
    print("Maximum size:", sizes.max())


========== Ring 1 ==========
count    1464.000000
mean        5.978142
std         1.497617
min         3.000000
25%         5.000000
50%         6.000000
75%         7.000000
max        13.000000
dtype: float64
Empty rings: 0
Minimum size: 3
Maximum size: 13

========== Ring 2 ==========
count    1464.000000
mean       13.938525
std         4.288508
min         6.000000
25%        11.000000
50%        13.000000
75%        16.000000
max        38.000000
dtype: float64
Empty rings: 0
Minimum size: 6
Maximum size: 38

========== Ring 3 ==========
count    1464.000000
mean       24.733607
std         8.045416
min        11.000000
25%        19.000000
50%        24.000000
75%        28.000000
max        83.000000
dtype: float64
Empty rings: 0
Minimum size: 11
Maximum size: 83


In [25]:
# Chapter 3.7 — Validate ring separation

overlap_12 = 0
overlap_13 = 0
overlap_23 = 0
self_in_ring = 0

for i in range(n_sites):
    r1 = rings[1][i]
    r2 = rings[2][i]
    r3 = rings[3][i]

    overlap_12 += len(r1 & r2)
    overlap_13 += len(r1 & r3)
    overlap_23 += len(r2 & r3)

    self_in_ring += int(
        i in r1 or
        i in r2 or
        i in r3
    )

print("Total Ring1 ∩ Ring2 memberships:", overlap_12)
print("Total Ring1 ∩ Ring3 memberships:", overlap_13)
print("Total Ring2 ∩ Ring3 memberships:", overlap_23)
print("Sites appearing in their own rings:", self_in_ring)

assert overlap_12 == 0
assert overlap_13 == 0
assert overlap_23 == 0
assert self_in_ring == 0

print("\n✓ Rings 1, 2, and 3 are mutually exclusive.")
print("✓ No focal site appears in its own spatial rings.")

Total Ring1 ∩ Ring2 memberships: 0
Total Ring1 ∩ Ring3 memberships: 0
Total Ring2 ∩ Ring3 memberships: 0
Sites appearing in their own rings: 0

✓ Rings 1, 2, and 3 are mutually exclusive.
✓ No focal site appears in its own spatial rings.


In [26]:
# Chapter 3.8 — Inspect representative spatial rings

for i in range(5):
    print(f"\nSite {i}")
    print(f"  Ring 1 ({len(rings[1][i]):2d}): {sorted(rings[1][i])}")
    print(f"  Ring 2 ({len(rings[2][i]):2d}): {sorted(rings[2][i])}")
    print(f"  Ring 3 ({len(rings[3][i]):2d}): {sorted(rings[3][i])}")


Site 0
  Ring 1 ( 5): [336, 998, 1200, 1230, 1277]
  Ring 2 (11): [48, 101, 265, 269, 344, 351, 529, 857, 879, 891, 1325]
  Ring 3 (18): [4, 98, 179, 247, 268, 459, 471, 588, 616, 694, 818, 826, 946, 967, 1172, 1191, 1447, 1457]

Site 1
  Ring 1 ( 5): [19, 229, 1067, 1097, 1173]
  Ring 2 (10): [105, 169, 200, 270, 900, 1019, 1130, 1139, 1305, 1321]
  Ring 3 (13): [37, 73, 159, 307, 427, 506, 524, 536, 708, 783, 881, 1119, 1324]

Site 2
  Ring 1 ( 5): [29, 240, 295, 1060, 1098]
  Ring 2 (10): [100, 337, 500, 586, 752, 822, 1018, 1357, 1383, 1422]
  Ring 3 (18): [64, 118, 160, 166, 178, 297, 420, 566, 634, 718, 766, 842, 1076, 1184, 1212, 1218, 1370, 1423]

Site 3
  Ring 1 ( 6): [90, 157, 323, 533, 794, 1329]
  Ring 2 (18): [12, 33, 92, 287, 569, 609, 662, 666, 736, 748, 773, 903, 934, 940, 1276, 1317, 1351, 1368]
  Ring 3 (29): [16, 47, 243, 250, 310, 319, 349, 371, 378, 413, 541, 574, 584, 659, 697, 785, 813, 843, 854, 860, 918, 982, 1012, 1181, 1189, 1278, 1327, 1361, 1425]

Site 4
 

## Chapter 4 — Linear spatial detrending before GCCM

The official polygon GCCM example removes the first-order spatial trend from both the candidate cause variable and the effect variable before spatial state-space reconstruction.

For the Seattle point-data adaptation, each site-level variable is therefore residualized using:

$$
Z_i = \beta_0 + \beta_x X_i + \beta_y Y_i + \varepsilon_i
$$

where \(X_i\) and \(Y_i\) are UTM coordinates.

The GCCM analysis uses the residual term:

$$
Z_i^{*} = Z_i - \hat{Z}_i
$$

This operation is performed only on the 1,464 unique spatial sites derived from Inner Fit.

The eight candidate causal attributes are:

- bathrooms
- sqft_living
- sqft_lot
- grade
- condition
- waterfront
- view
- age

House price is treated as the effect variable.

Coordinate centering is used only for numerical stability. With an intercept in the regression, centering the UTM coordinates does not change the fitted linear spatial trend.

In [27]:
# Chapter 4.1 — Remove first-order linear spatial trend

import numpy as np
import pandas as pd

site_variables = causal_feature_cols + ["price"]

# Site-level coordinates
site_xy = gccm_inner_sites[coord_cols].to_numpy(dtype=float)

# Center coordinates for numerical stability.
# This does NOT change the fitted OLS values when an intercept is included.
site_xy_centered = site_xy - site_xy.mean(axis=0)

design = np.column_stack([
    np.ones(len(site_xy_centered)),
    site_xy_centered
])

gccm_inner_detrended = gccm_inner_sites.copy()

detrend_diagnostics = []

for var in site_variables:

    z = gccm_inner_sites[var].to_numpy(dtype=float)

    # OLS: z ~ 1 + UTM_X + UTM_Y
    beta, *_ = np.linalg.lstsq(
        design,
        z,
        rcond=None
    )

    fitted = design @ beta
    residual = z - fitted

    residual_col = f"{var}_resid"

    gccm_inner_detrended[residual_col] = residual

    ss_total = np.sum((z - z.mean()) ** 2)
    ss_resid = np.sum(residual ** 2)

    trend_r2 = (
        1 - ss_resid / ss_total
        if ss_total > 0
        else np.nan
    )

    detrend_diagnostics.append({
        "variable": var,
        "raw_mean": z.mean(),
        "raw_std": z.std(ddof=1),
        "residual_mean": residual.mean(),
        "residual_std": residual.std(ddof=1),
        "spatial_trend_R2": trend_r2
    })

detrend_diagnostics = pd.DataFrame(detrend_diagnostics)

print("Site-level observations:", len(gccm_inner_detrended))
print("\nDetrending diagnostics:")
display(detrend_diagnostics.round(6))

Site-level observations: 1464

Detrending diagnostics:


,variable,raw_mean,raw_std,residual_mean,residual_std,spatial_trend_R2
0,bathrooms,2.068818,0.754899,-0.0,0.742189,0.033389
1,sqft_living,1995.477459,845.255839,0.0,821.947600,0.054390
2,sqft_lot,7596.650615,3745.072740,0.0,3480.557785,0.136272
3,grade,7.524932,1.094428,-0.0,1.070096,0.043971
4,condition,3.452186,0.677059,-0.0,0.676823,0.000699
5,waterfront,0.005464,0.073745,0.0,0.073695,0.001369
6,view,0.235656,0.750347,-0.0,0.746136,0.011191
7,age,48.204577,29.068190,-0.0,27.305083,0.117629
8,price,53.138750,37.556525,0.0,35.783657,0.092182


In [28]:
# Chapter 4.2 — Audit detrended variables

residual_cols = [
    f"{var}_resid"
    for var in site_variables
]

print("Finite-value check:")
print(
    gccm_inner_detrended[residual_cols]
    .apply(lambda s: np.isfinite(s).all())
)

print("\nResidual means:")
for var in site_variables:

    r = gccm_inner_detrended[f"{var}_resid"].to_numpy()

    print(
        f"{var:15s} "
        f"mean={r.mean(): .3e} "
        f"std={r.std(ddof=1): .6f}"
    )

Finite-value check:
bathrooms_resid      True
sqft_living_resid    True
sqft_lot_resid       True
grade_resid          True
condition_resid      True
waterfront_resid     True
view_resid           True
age_resid            True
price_resid          True
dtype: bool

Residual means:
bathrooms       mean=-1.323e-15 std= 0.742189
sqft_living     mean= 1.712e-12 std= 821.947600
sqft_lot        mean= 7.992e-12 std= 3480.557785
grade           mean=-3.326e-14 std= 1.070096
condition       mean=-2.998e-15 std= 0.676823
waterfront      mean= 1.820e-17 std= 0.073695
view            mean=-2.269e-16 std= 0.746136
age             mean=-3.479e-14 std= 27.305083
price           mean= 5.999e-14 std= 35.783657


In [29]:
# Chapter 4.3 — Check orthogonality to the linear spatial coordinates

utm_x = site_xy[:, 0]
utm_y = site_xy[:, 1]

spatial_residual_check = []

for var in site_variables:

    r = gccm_inner_detrended[
        f"{var}_resid"
    ].to_numpy(dtype=float)

    corr_x = np.corrcoef(r, utm_x)[0, 1]
    corr_y = np.corrcoef(r, utm_y)[0, 1]

    spatial_residual_check.append({
        "variable": var,
        "corr_resid_UTM_X": corr_x,
        "corr_resid_UTM_Y": corr_y
    })

spatial_residual_check = pd.DataFrame(
    spatial_residual_check
)

display(
    spatial_residual_check.round(12)
)

max_abs_corr = np.nanmax(
    np.abs(
        spatial_residual_check[
            ["corr_resid_UTM_X",
             "corr_resid_UTM_Y"]
        ].to_numpy()
    )
)

print(
    "\nMaximum absolute residual-coordinate correlation:",
    f"{max_abs_corr:.3e}"
)

assert np.isfinite(
    gccm_inner_detrended[residual_cols].to_numpy()
).all()

assert all(
    gccm_inner_detrended[col].std(ddof=1) > 0
    for col in residual_cols
)

print("\n✓ All detrended variables are finite.")
print("✓ All detrended variables retain non-zero variation.")

,variable,corr_resid_UTM_X,corr_resid_UTM_Y
0,bathrooms,-0.0,-0.0
1,sqft_living,-0.0,0.0
2,sqft_lot,-0.0,-0.0
3,grade,0.0,-0.0
4,condition,0.0,-0.0
5,waterfront,-0.0,0.0
6,view,0.0,0.0
7,age,0.0,-0.0
8,price,-0.0,0.0



Maximum absolute residual-coordinate correlation: 6.324e-16

✓ All detrended variables are finite.
✓ All detrended variables retain non-zero variation.


In [30]:
# Chapter 4.4 — Inspect discrete / low-cardinality causal attributes

discrete_features = [
    "waterfront",
    "view",
    "condition",
    "grade"
]

for var in discrete_features:

    print("\n" + "=" * 60)
    print(var)

    counts = (
        gccm_inner_sites[var]
        .value_counts()
        .sort_index()
    )

    print("Unique values:", gccm_inner_sites[var].nunique())
    print(counts)


waterfront
Unique values: 2
waterfront
0.0    1456
1.0       8
Name: count, dtype: int64

view
Unique values: 5
view
0.0    1311
1.0      32
2.0      67
3.0      37
4.0      17
Name: count, dtype: int64

condition
Unique values: 7
condition
1.0      2
2.0     13
2.5      1
3.0    902
3.5      3
4.0    408
5.0    135
Name: count, dtype: int64

grade
Unique values: 12
grade
4.0       1
5.0      20
6.0     150
6.5       2
7.0     657
7.5       1
8.0     412
9.0     145
10.0     50
11.0     22
12.0      2
13.0      2
Name: count, dtype: int64


## Chapter 5 — Exporting Seattle point-GCCM inputs to R

The Seattle GCCM causal-estimation dataset is now frozen at the Inner-Fit level:

- 1,470 record-level observations;
- 1,464 unique spatial sites after within-subset aggregation;
- Delaunay point adjacency;
- embedding dimension \(E=3\);
- eight candidate housing attributes and house price;
- first-order UTM spatial trends removed from all GCCM variables.

Python is used for experiment management, data splitting, site aggregation, and point-graph construction.

The released official GCCM R implementation is retained for spatial embedding and cross mapping.
To preserve the exact site ordering, each site is assigned a 1-based `site_id` before export.

In [31]:
# Chapter 5.1 — Export site-level GCCM data

from pathlib import Path
import numpy as np
import pandas as pd

gccm_export_dir = Path(GCCM_OUTPUT_DIR)
gccm_export_dir.mkdir(parents=True, exist_ok=True)

sites_export = gccm_inner_detrended.copy()

# R uses 1-based indices, so assign explicit 1-based site IDs.
sites_export.insert(
    0,
    "site_id",
    np.arange(1, len(sites_export) + 1)
)

sites_path = (
    gccm_export_dir
    / "seattle_gccm_inner_fit_sites.csv"
)

sites_export.to_csv(
    sites_path,
    index=False
)

print("Saved:")
print(sites_path)

print("\nShape:", sites_export.shape)
print("\nFirst columns:")
print(sites_export.columns.tolist())

print("\nFirst 3 rows:")
display(sites_export.head(3))

Saved:
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccm\seattle_gccm_inner_fit_sites.csv

Shape: (1464, 21)

First columns:
['site_id', 'UTM_X', 'UTM_Y', 'bathrooms', 'sqft_living', 'sqft_lot', 'grade', 'condition', 'waterfront', 'view', 'age', 'price', 'bathrooms_resid', 'sqft_living_resid', 'sqft_lot_resid', 'grade_resid', 'condition_resid', 'waterfront_resid', 'view_resid', 'age_resid', 'price_resid']

First 3 rows:


,site_id,UTM_X,UTM_Y,bathrooms,sqft_living,sqft_lot,grade,condition,waterfront,view,...,price,bathrooms_resid,sqft_living_resid,sqft_lot_resid,grade_resid,condition_resid,waterfront_resid,view_resid,age_resid,price_resid
0,1,552525.572916,5.282439e+06,2.25,2440.0,9889.0,7.0,3.0,0.0,0.0,...,80.500,0.207982,477.094757,3082.541970,-0.534343,-0.453597,-0.005210,-0.283384,8.040025,18.948263
1,2,554693.758122,5.267287e+06,1.00,1150.0,6835.0,7.0,3.0,0.0,1.0,...,42.500,-1.054301,-823.980420,-704.235531,-0.494571,-0.449753,-0.005143,0.761562,44.626905,-8.894468
2,3,550231.670995,5.261212e+06,1.00,1050.0,6250.0,6.0,4.0,0.0,0.0,...,20.995,-0.947406,-769.106657,-538.413055,-1.302407,0.565328,-0.002937,-0.279348,19.296075,-23.134186


In [32]:
# Chapter 5.2 — Export Delaunay adjacency as a directed edge table

edge_rows = []

for i, neighs in enumerate(delaunay_neighbors_0based):
    for j in neighs:
        edge_rows.append({
            "from_site": i + 1,
            "to_site": int(j) + 1
        })

delaunay_edges = pd.DataFrame(edge_rows)

edges_path = (
    gccm_export_dir
    / "seattle_gccm_delaunay_edges.csv"
)

delaunay_edges.to_csv(
    edges_path,
    index=False
)

print("Saved:")
print(edges_path)

print("\nDirected adjacency rows:", len(delaunay_edges))
print("Expected:", 2 * 4376)

print("\nUnique from-sites:",
      delaunay_edges["from_site"].nunique())

print("Unique to-sites:",
      delaunay_edges["to_site"].nunique())

display(delaunay_edges.head(10))

assert len(delaunay_edges) == 2 * 4376
assert delaunay_edges["from_site"].nunique() == 1464
assert delaunay_edges["to_site"].nunique() == 1464

print("\n✓ Delaunay adjacency exported with 1-based R indices.")

Saved:
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccm\seattle_gccm_delaunay_edges.csv

Directed adjacency rows: 8752
Expected: 8752

Unique from-sites: 1464
Unique to-sites: 1464


,from_site,to_site
0,1,337
1,1,999
2,1,1201
3,1,1231
4,1,1278
5,2,20
6,2,230
7,2,1068
8,2,1098
9,2,1174



✓ Delaunay adjacency exported with 1-based R indices.


In [33]:
# Chapter 5.3 — Save preprocessing diagnostics

detrend_path = (
    gccm_export_dir
    / "seattle_gccm_detrend_diagnostics.csv"
)

detrend_diagnostics.to_csv(
    detrend_path,
    index=False
)

ring_summary = pd.DataFrame({
    "ring": [1, 2, 3],
    "mean_size": [
        np.mean([len(x) for x in rings[1]]),
        np.mean([len(x) for x in rings[2]]),
        np.mean([len(x) for x in rings[3]])
    ],
    "min_size": [
        min(len(x) for x in rings[1]),
        min(len(x) for x in rings[2]),
        min(len(x) for x in rings[3])
    ],
    "max_size": [
        max(len(x) for x in rings[1]),
        max(len(x) for x in rings[2]),
        max(len(x) for x in rings[3])
    ],
    "empty_rings": [
        sum(len(x) == 0 for x in rings[1]),
        sum(len(x) == 0 for x in rings[2]),
        sum(len(x) == 0 for x in rings[3])
    ]
})

ring_path = (
    gccm_export_dir
    / "seattle_gccm_ring_diagnostics.csv"
)

ring_summary.to_csv(
    ring_path,
    index=False
)

print("Saved diagnostics:")
print(detrend_path)
print(ring_path)

display(ring_summary)

Saved diagnostics:
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccm\seattle_gccm_detrend_diagnostics.csv
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccm\seattle_gccm_ring_diagnostics.csv


,ring,mean_size,min_size,max_size,empty_rings
0,1,5.978142,3,13,0
1,2,13.938525,6,38,0
2,3,24.733607,11,83,0


## Chapter 6 — Inner-Fit GCCM causal strengths

For the directional hypothesis

$$
X_j \rightarrow Price,
$$

the corresponding GCCM cross-map is

$$
Price \; xmap \; X_j.
$$

A candidate feature is retained as causally informative when:

1. the terminal cross-map skill exceeds the initial skill;
2. the Spearman correlation between library size and cross-map skill is at least 0.60;
3. the terminal two-sided significance test gives \(p<0.05\);
4. the terminal 95% confidence interval has a lower bound greater than zero.

For retained features, causal strength is defined as the terminal cross-map skill:

$$
s_j=\rho_j(L_{\max}).
$$

Features failing the convergence criteria receive zero strength.

The normalized causal feature weight is

$$
w_j=\frac{s_j}{\sum_k s_k}.
$$

These Inner-Fit weights are used only for GCCP hyperparameter development.
They are not estimated from Inner Calibration, Inner Validation, Outer Calibration, or Test data.

In [37]:
# Chapter 6.1 — Build and freeze Inner-Fit GCCM causal weights

import numpy as np
import pandas as pd
from pathlib import Path

diag_path = (
    Path(GCCM_OUTPUT_DIR)
    / "seattle_gccm_feature_to_price_convergence_diagnostics.csv"
)

gccm_diag = pd.read_csv(diag_path)

# Preserve the exact feature order used by GeoSIMCP / GCCP
feature_order = [
    "bathrooms",
    "sqft_living",
    "sqft_lot",
    "grade",
    "condition",
    "waterfront",
    "view",
    "age"
]

gccm_diag["causal_strength"] = np.where(
    gccm_diag["converged"].astype(bool),
    gccm_diag["rho_Lmax"],
    0.0
)

strength_sum = gccm_diag["causal_strength"].sum()

assert strength_sum > 0

gccm_diag["causal_weight"] = (
    gccm_diag["causal_strength"] / strength_sum
)

gccm_weights_inner = (
    gccm_diag
    .set_index("feature")
    .loc[feature_order]
    .reset_index()
)

print("Inner-Fit GCCM causal weights")
print("=" * 60)

display(
    gccm_weights_inner[
        [
            "feature",
            "rho_L10",
            "rho_Lmax",
            "gain",
            "spearman",
            "p_two_sided",
            "CI_lower",
            "CI_upper",
            "converged",
            "causal_strength",
            "causal_weight"
        ]
    ].round(6)
)

print("\nStrength sum:", strength_sum)
print("Weight sum:", gccm_weights_inner["causal_weight"].sum())

assert np.isclose(
    gccm_weights_inner["causal_weight"].sum(),
    1.0
)

assert (
    gccm_weights_inner.loc[
        gccm_weights_inner["feature"].isin(["age", "waterfront"]),
        "causal_weight"
    ] == 0
).all()

print("\n✓ Inner-Fit GCCM causal weights frozen.")

Inner-Fit GCCM causal weights


,feature,rho_L10,rho_Lmax,gain,spearman,p_two_sided,CI_lower,CI_upper,converged,causal_strength,causal_weight
0,bathrooms,0.074023,0.193219,0.119196,1.000000,0.000000,0.143372,0.242088,True,0.193219,0.158578
1,sqft_living,0.135054,0.267773,0.132720,1.000000,0.000000,0.219520,0.314720,True,0.267773,0.219766
2,sqft_lot,0.049122,0.201288,0.152166,1.000000,0.000000,0.151585,0.249975,True,0.201288,0.165200
3,grade,0.204869,0.315271,0.110403,1.000000,0.000000,0.268341,0.360708,True,0.315271,0.258749
4,condition,0.020960,0.089394,0.068434,1.000000,0.000622,0.038302,0.140019,True,0.089394,0.073367
5,waterfront,0.029616,0.000000,-0.029616,-0.942857,1.000000,-0.051267,0.051267,False,0.000000,0.000000
6,view,0.082664,0.151501,0.068837,0.657143,0.000000,0.101018,0.201205,True,0.151501,0.124340
7,age,0.009838,0.041021,0.031182,1.000000,0.116931,-0.010268,0.092094,False,0.000000,0.000000



Strength sum: 1.2184458501734619
Weight sum: 1.0

✓ Inner-Fit GCCM causal weights frozen.


In [38]:
# Chapter 6.2 — Save frozen Inner-Fit GCCM weights

from pathlib import Path

weights_path = (
    Path(GCCM_OUTPUT_DIR)
    / "seattle_gccm_causal_weights_inner_fit_v1.csv"
)

gccm_weights_inner.to_csv(
    weights_path,
    index=False
)

print("Saved:")
print(weights_path)

print("\n✓ Chapter 6 Inner-Fit causal structure complete.")

Saved:
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccm\seattle_gccm_causal_weights_inner_fit_v1.csv

✓ Chapter 6 Inner-Fit causal structure complete.


## Chapter 7 — Final GCCM causal structure using Outer Train

After GCCP hyperparameter tuning is completed using the Inner experiment,
the causal structure is re-estimated using the complete Outer Train set.

No Outer Calibration or Outer Test observations are used for causal
structure estimation.

The complete GCCM procedure is kept identical to the Inner-Fit stage:

1. aggregate duplicated coordinates within Outer Train only;
2. construct a Delaunay spatial graph;
3. use embedding dimension \(E=3\);
4. remove first-order linear UTM spatial trends;
5. evaluate \(Price\ xmap\ X_j\);
6. assess convergence using the same pre-specified criteria;
7. use terminal cross-map skill as causal strength for retained features.

These Outer-Train causal weights are frozen before evaluation on the
Outer Test set.

In [40]:
# Chapter 7.1 — Build Outer-Train site-level GCCM dataset
# Use the Outer-Train variables already restored in this notebook

import numpy as np
import pandas as pd

print("===== OUTER TRAIN INPUT CHECK =====")

print(
    "X causal:",
    X_causal_outer_train.shape
)

print(
    "Coordinates:",
    coordinates_outer_train.shape
)

print(
    "Target:",
    y_outer_train.shape
)

# ---------------------------------------------------------
# 1. Reset indices so the three objects align row-by-row
# ---------------------------------------------------------

X_outer = (
    X_causal_outer_train
    .reset_index(drop=True)
    .copy()
)

coords_outer = (
    coordinates_outer_train
    .reset_index(drop=True)
    .copy()
)

y_outer = (
    pd.Series(y_outer_train)
    .reset_index(drop=True)
    .astype(float)
)

assert len(X_outer) == 2100
assert len(coords_outer) == 2100
assert len(y_outer) == 2100

# ---------------------------------------------------------
# 2. Assemble record-level GCCM table
# ---------------------------------------------------------

outer_records = pd.concat(
    [
        coords_outer[["UTM_X", "UTM_Y"]],
        X_outer[causal_feature_cols]
    ],
    axis=1
)

outer_records["price"] = (
    y_outer.to_numpy(dtype=float)
)

print(
    "\nOuter Train record-level shape:",
    outer_records.shape
)

# ---------------------------------------------------------
# 3. Aggregate duplicate coordinates WITHIN Outer Train only
# ---------------------------------------------------------

gccm_outer_sites = (
    outer_records
    .groupby(
        ["UTM_X", "UTM_Y"],
        sort=False,
        as_index=False
    )
    .mean(numeric_only=True)
)

print(
    "Outer Train unique-site shape:",
    gccm_outer_sites.shape
)

print(
    "Records aggregated:",
    len(outer_records)
    - len(gccm_outer_sites)
)

duplicate_coords = (
    gccm_outer_sites
    .duplicated(
        subset=["UTM_X", "UTM_Y"]
    )
    .sum()
)

print(
    "Remaining duplicated coordinates:",
    duplicate_coords
)

print(
    "Missing values:",
    gccm_outer_sites.isna().sum().sum()
)

# ---------------------------------------------------------
# 4. Integrity checks
# ---------------------------------------------------------

assert len(outer_records) == 2100
assert duplicate_coords == 0
assert gccm_outer_sites.isna().sum().sum() == 0

print(
    "\n✓ Outer Train site-level GCCM dataset ready."
)

===== OUTER TRAIN INPUT CHECK =====
X causal: (2100, 8)
Coordinates: (2100, 2)
Target: (2100,)

Outer Train record-level shape: (2100, 11)
Outer Train unique-site shape: (2088, 11)
Records aggregated: 12
Remaining duplicated coordinates: 0
Missing values: 0

✓ Outer Train site-level GCCM dataset ready.


### 7.2 Outer-Train Delaunay spatial graph

The final GCCM spatial graph is reconstructed using the complete
Outer Train set only.

After within-subset aggregation, the 2,100 Outer-Train records correspond
to 2,088 unique spatial sites.

The same point-neighborhood definition used in the Inner experiment is
retained:

- Delaunay triangulation;
- undirected symmetric adjacency;
- no self-neighbors;
- embedding dimension \(E=3\).

The graph construction rule is therefore unchanged between Inner tuning
and final causal-structure estimation.

In [41]:
# Chapter 7.2 — Build Outer-Train Delaunay graph

import numpy as np
import pandas as pd

from scipy.spatial import Delaunay

# ---------------------------------------------------------
# 1. Coordinates
# ---------------------------------------------------------

site_coords_outer = (
    gccm_outer_sites[
        ["UTM_X", "UTM_Y"]
    ]
    .to_numpy(dtype=float)
)

n_sites_outer = len(site_coords_outer)

print(
    "Outer Train unique spatial sites:",
    n_sites_outer
)

assert n_sites_outer == 2088
assert np.isfinite(site_coords_outer).all()

# ---------------------------------------------------------
# 2. Delaunay triangulation
# ---------------------------------------------------------

tri_outer = Delaunay(
    site_coords_outer
)

neighbor_sets_outer = [
    set()
    for _ in range(n_sites_outer)
]

edge_set_outer = set()

for triangle in tri_outer.simplices:

    a, b, c = map(int, triangle)

    for i, j in [
        (a, b),
        (b, c),
        (c, a)
    ]:

        if i == j:
            continue

        u, v = sorted((i, j))

        edge_set_outer.add(
            (u, v)
        )

        neighbor_sets_outer[i].add(j)
        neighbor_sets_outer[j].add(i)

delaunay_neighbors_outer_0based = [
    np.array(
        sorted(neigh),
        dtype=int
    )
    for neigh in neighbor_sets_outer
]

degrees_outer = np.array(
    [
        len(neigh)
        for neigh in neighbor_sets_outer
    ],
    dtype=int
)

# ---------------------------------------------------------
# 3. Graph diagnostics
# ---------------------------------------------------------

print(
    "\nDelaunay triangles:",
    len(tri_outer.simplices)
)

print(
    "Undirected edges:",
    len(edge_set_outer)
)

print("\nDegree diagnostics:")

degree_summary_outer = pd.Series(
    degrees_outer
).describe(
    percentiles=[0.25, 0.50, 0.75]
)

display(
    degree_summary_outer
    .to_frame("degree")
)

zero_neighbor_sites = np.sum(
    degrees_outer == 0
)

print(
    "Zero-neighbor sites:",
    zero_neighbor_sites
)

# ---------------------------------------------------------
# 4. Symmetry / self-neighbor checks
# ---------------------------------------------------------

symmetric_outer = all(
    i in neighbor_sets_outer[j]
    for i in range(n_sites_outer)
    for j in neighbor_sets_outer[i]
)

self_neighbor_outer = any(
    i in neighbor_sets_outer[i]
    for i in range(n_sites_outer)
)

print(
    "Adjacency symmetric:",
    symmetric_outer
)

print(
    "Any self-neighbors:",
    self_neighbor_outer
)

assert zero_neighbor_sites == 0
assert symmetric_outer
assert not self_neighbor_outer

print(
    "\n✓ Outer-Train Delaunay graph passed structural checks."
)

Outer Train unique spatial sites: 2088

Delaunay triangles: 4158
Undirected edges: 6245

Degree diagnostics:


,degree
count,2088.000000
mean,5.981801
std,1.539032
min,3.000000
25%,5.000000
50%,6.000000
75%,7.000000
max,13.000000


Zero-neighbor sites: 0
Adjacency symmetric: True
Any self-neighbors: False

✓ Outer-Train Delaunay graph passed structural checks.


In [42]:
# Chapter 7.3 — Audit exact graph-distance rings for E = 3

def exact_graph_rings(
    neighbor_sets,
    max_ring=3
):
    """
    Return exact shortest-path graph-distance rings.

    rings[k][i] = nodes whose shortest graph distance
                  from focal node i is exactly k.
    """

    n = len(neighbor_sets)

    rings = {
        k: []
        for k in range(1, max_ring + 1)
    }

    for focal in range(n):

        visited = {focal}
        frontier = {focal}

        for k in range(1, max_ring + 1):

            next_frontier = set()

            for node in frontier:
                next_frontier.update(
                    neighbor_sets[node]
                )

            next_frontier -= visited

            rings[k].append(
                np.array(
                    sorted(next_frontier),
                    dtype=int
                )
            )

            visited.update(
                next_frontier
            )

            frontier = next_frontier

    return rings


rings_outer = exact_graph_rings(
    neighbor_sets_outer,
    max_ring=3
)

ring_diagnostics_outer = []

for k in [1, 2, 3]:

    sizes = np.array(
        [
            len(r)
            for r in rings_outer[k]
        ]
    )

    ring_diagnostics_outer.append({
        "ring": k,
        "mean_size": sizes.mean(),
        "std_size": sizes.std(ddof=1),
        "min_size": sizes.min(),
        "q1": np.quantile(sizes, 0.25),
        "median": np.median(sizes),
        "q3": np.quantile(sizes, 0.75),
        "max_size": sizes.max(),
        "empty_rings": np.sum(sizes == 0),
    })

ring_diagnostics_outer = pd.DataFrame(
    ring_diagnostics_outer
)

display(
    ring_diagnostics_outer.round(6)
)

# ---------------------------------------------------------
# Ring-overlap and focal-self checks
# ---------------------------------------------------------

overlap_12 = 0
overlap_13 = 0
overlap_23 = 0
self_in_ring = 0

for i in range(n_sites_outer):

    r1 = set(rings_outer[1][i])
    r2 = set(rings_outer[2][i])
    r3 = set(rings_outer[3][i])

    overlap_12 += len(r1 & r2)
    overlap_13 += len(r1 & r3)
    overlap_23 += len(r2 & r3)

    self_in_ring += int(
        i in r1
        or i in r2
        or i in r3
    )

print(
    "\nRing 1 / Ring 2 overlap:",
    overlap_12
)

print(
    "Ring 1 / Ring 3 overlap:",
    overlap_13
)

print(
    "Ring 2 / Ring 3 overlap:",
    overlap_23
)

print(
    "Focal self appearing in any ring:",
    self_in_ring
)

assert overlap_12 == 0
assert overlap_13 == 0
assert overlap_23 == 0
assert self_in_ring == 0

assert (
    ring_diagnostics_outer[
        "empty_rings"
    ] == 0
).all()

print(
    "\n✓ Ring 1/2/3 audit passed."
)
print(
    "✓ E = 3 remains valid for Final Outer-Train GCCM."
)

max_library_outer = (
    n_sites_outer - (3 - 1)
)

print(
    "\nFinal GCCM maximum library size:",
    max_library_outer
)

,ring,mean_size,std_size,min_size,q1,median,q3,max_size,empty_rings
0,1,5.981801,1.539032,3,5.0,6.0,7.0,13,0
1,2,14.078544,4.229356,5,11.0,13.0,16.0,49,0
2,3,25.001916,8.200354,10,20.0,24.0,28.0,92,0



Ring 1 / Ring 2 overlap: 0
Ring 1 / Ring 3 overlap: 0
Ring 2 / Ring 3 overlap: 0
Focal self appearing in any ring: 0

✓ Ring 1/2/3 audit passed.
✓ E = 3 remains valid for Final Outer-Train GCCM.

Final GCCM maximum library size: 2086


### 7.4 Remove first-order spatial trends on Outer Train

The same detrending procedure used during Inner-Fit GCCM estimation is
applied to the complete Outer Train site-level dataset.

For each candidate feature and house price,

$$
Z_i
=
\beta_0
+
\beta_x UTM_{X,i}
+
\beta_y UTM_{Y,i}
+
\varepsilon_i.
$$

GCCM is performed using the residual component

$$
Z_i^*
=
Z_i-\hat Z_i.
$$

Spatial detrending is estimated using Outer Train only.
Outer Calibration and Outer Test are not involved.

In [43]:
# Chapter 7.4 — Detrend Outer-Train GCCM variables

import numpy as np
import pandas as pd

site_variables_outer = (
    causal_feature_cols
    + ["price"]
)

site_xy_outer = (
    gccm_outer_sites[
        ["UTM_X", "UTM_Y"]
    ]
    .to_numpy(dtype=float)
)

# Center coordinates for numerical stability
site_xy_outer_centered = (
    site_xy_outer
    - site_xy_outer.mean(axis=0)
)

design_outer = np.column_stack([
    np.ones(len(site_xy_outer_centered)),
    site_xy_outer_centered
])

gccm_outer_detrended = (
    gccm_outer_sites.copy()
)

detrend_diagnostics_outer = []

for var in site_variables_outer:

    z = gccm_outer_sites[
        var
    ].to_numpy(dtype=float)

    beta, *_ = np.linalg.lstsq(
        design_outer,
        z,
        rcond=None
    )

    fitted = design_outer @ beta
    residual = z - fitted

    gccm_outer_detrended[
        f"{var}_resid"
    ] = residual

    ss_total = np.sum(
        (z - z.mean()) ** 2
    )

    ss_resid = np.sum(
        residual ** 2
    )

    trend_r2 = (
        1 - ss_resid / ss_total
        if ss_total > 0
        else np.nan
    )

    detrend_diagnostics_outer.append({
        "variable": var,
        "raw_mean": z.mean(),
        "raw_std": z.std(ddof=1),
        "residual_mean": residual.mean(),
        "residual_std": residual.std(ddof=1),
        "spatial_trend_R2": trend_r2
    })

detrend_diagnostics_outer = pd.DataFrame(
    detrend_diagnostics_outer
)

print(
    "Outer-Train site observations:",
    len(gccm_outer_detrended)
)

display(
    detrend_diagnostics_outer.round(6)
)

Outer-Train site observations: 2088


,variable,raw_mean,raw_std,residual_mean,residual_std,spatial_trend_R2
0,bathrooms,2.053400,0.764773,-0.0,0.750506,0.036961
1,sqft_living,1974.029215,860.934689,0.0,834.837177,0.059707
2,sqft_lot,7460.718151,3716.876372,0.0,3435.312990,0.145767
3,grade,7.514847,1.119916,-0.0,1.092752,0.047922
4,condition,3.444205,0.683592,0.0,0.683321,0.000791
5,waterfront,0.005268,0.072408,0.0,0.072400,0.000237
6,view,0.246648,0.770807,-0.0,0.765958,0.012540
7,age,48.553161,29.421369,0.0,27.608672,0.119427
8,price,53.183291,37.554896,0.0,35.764735,0.093063


In [44]:
# Chapter 7.5 — Validate Outer-Train detrending

residual_cols_outer = [
    f"{v}_resid"
    for v in site_variables_outer
]

# Finite and non-zero variance checks
assert np.isfinite(
    gccm_outer_detrended[
        residual_cols_outer
    ].to_numpy()
).all()

assert all(
    gccm_outer_detrended[col].std(ddof=1) > 0
    for col in residual_cols_outer
)

utm_x_outer = site_xy_outer[:, 0]
utm_y_outer = site_xy_outer[:, 1]

spatial_check_outer = []

for var in site_variables_outer:

    r = gccm_outer_detrended[
        f"{var}_resid"
    ].to_numpy(dtype=float)

    spatial_check_outer.append({
        "variable": var,
        "corr_resid_UTM_X":
            np.corrcoef(r, utm_x_outer)[0, 1],
        "corr_resid_UTM_Y":
            np.corrcoef(r, utm_y_outer)[0, 1]
    })

spatial_check_outer = pd.DataFrame(
    spatial_check_outer
)

display(
    spatial_check_outer.round(12)
)

max_abs_corr_outer = np.nanmax(
    np.abs(
        spatial_check_outer[
            [
                "corr_resid_UTM_X",
                "corr_resid_UTM_Y"
            ]
        ].to_numpy()
    )
)

print(
    "\nMaximum absolute residual-coordinate correlation:",
    f"{max_abs_corr_outer:.3e}"
)

print(
    "\n✓ All Outer-Train detrended variables are finite."
)
print(
    "✓ All retain non-zero variation."
)

,variable,corr_resid_UTM_X,corr_resid_UTM_Y
0,bathrooms,-0.0,-0.0
1,sqft_living,-0.0,-0.0
2,sqft_lot,-0.0,0.0
3,grade,-0.0,0.0
4,condition,0.0,-0.0
5,waterfront,0.0,-0.0
6,view,0.0,0.0
7,age,0.0,0.0
8,price,-0.0,-0.0



Maximum absolute residual-coordinate correlation: 8.963e-16

✓ All Outer-Train detrended variables are finite.
✓ All retain non-zero variation.


In [45]:
# Chapter 7.6 — Export Outer-Train GCCM site data

from pathlib import Path

gccm_export_dir = Path(
    GCCM_OUTPUT_DIR
)

gccm_export_dir.mkdir(
    parents=True,
    exist_ok=True
)

sites_outer_export = (
    gccm_outer_detrended.copy()
)

sites_outer_export.insert(
    0,
    "site_id",
    np.arange(
        1,
        len(sites_outer_export) + 1
    )
)

outer_sites_path = (
    gccm_export_dir
    / "seattle_gccm_outer_train_sites.csv"
)

sites_outer_export.to_csv(
    outer_sites_path,
    index=False
)

print("Saved:")
print(outer_sites_path)

print(
    "\nShape:",
    sites_outer_export.shape
)

print(
    "Residual columns:",
    sum(
        c.endswith("_resid")
        for c in sites_outer_export.columns
    )
)

display(
    sites_outer_export.head(3)
)

Saved:
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccm\seattle_gccm_outer_train_sites.csv

Shape: (2088, 21)
Residual columns: 9


,site_id,UTM_X,UTM_Y,bathrooms,sqft_living,sqft_lot,grade,condition,waterfront,view,...,price,bathrooms_resid,sqft_living_resid,sqft_lot_resid,grade_resid,condition_resid,waterfront_resid,view_resid,age_resid,price_resid
0,1,562301.996691,5.266441e+06,2.25,1640.0,8400.0,8.0,4.0,0.0,0.0,...,51.50,0.045954,-548.781242,-564.392566,0.254102,0.535960,-0.006298,-0.154731,15.042109,-3.588032
1,2,550868.718042,5.239800e+06,2.25,1715.0,6819.0,7.0,3.0,0.0,0.0,...,25.75,0.368468,-8.825843,-480.753655,-0.146833,-0.422489,-0.003537,-0.261767,-28.073146,-2.250943
2,3,549506.978421,5.239933e+06,2.50,2550.0,8170.0,8.0,3.0,0.0,0.0,...,31.60,0.647842,868.115163,1143.225180,0.899577,-0.418633,-0.003329,-0.278522,-19.975633,4.269662


In [46]:
# Chapter 7.7 — Export Outer-Train Delaunay adjacency

edge_rows_outer = []

for i, neighs in enumerate(
    delaunay_neighbors_outer_0based
):

    for j in neighs:

        edge_rows_outer.append({
            "from_site": i + 1,
            "to_site": int(j) + 1
        })

delaunay_edges_outer = pd.DataFrame(
    edge_rows_outer
)

outer_edges_path = (
    gccm_export_dir
    / "seattle_gccm_outer_train_delaunay_edges.csv"
)

delaunay_edges_outer.to_csv(
    outer_edges_path,
    index=False
)

print("Saved:")
print(outer_edges_path)

print(
    "\nDirected adjacency rows:",
    len(delaunay_edges_outer)
)

print(
    "Expected:",
    2 * 6245
)

print(
    "Unique from-sites:",
    delaunay_edges_outer[
        "from_site"
    ].nunique()
)

print(
    "Unique to-sites:",
    delaunay_edges_outer[
        "to_site"
    ].nunique()
)

assert len(
    delaunay_edges_outer
) == 2 * 6245

assert (
    delaunay_edges_outer[
        "from_site"
    ].nunique()
    == 2088
)

assert (
    delaunay_edges_outer[
        "to_site"
    ].nunique()
    == 2088
)

print(
    "\n✓ Final Outer-Train adjacency exported."
)

Saved:
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccm\seattle_gccm_outer_train_delaunay_edges.csv

Directed adjacency rows: 12490
Expected: 12490
Unique from-sites: 2088
Unique to-sites: 2088

✓ Final Outer-Train adjacency exported.


In [47]:
# Chapter 7.8 — Save Final GCCM preprocessing diagnostics

outer_detrend_path = (
    gccm_export_dir
    / "seattle_gccm_outer_train_detrend_diagnostics.csv"
)

outer_ring_path = (
    gccm_export_dir
    / "seattle_gccm_outer_train_ring_diagnostics.csv"
)

detrend_diagnostics_outer.to_csv(
    outer_detrend_path,
    index=False
)

ring_diagnostics_outer.to_csv(
    outer_ring_path,
    index=False
)

print("Saved:")
print(outer_detrend_path)
print(outer_ring_path)

print(
    "\n✓ Final GCCM preprocessing files saved."
)

Saved:
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccm\seattle_gccm_outer_train_detrend_diagnostics.csv
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccm\seattle_gccm_outer_train_ring_diagnostics.csv

✓ Final GCCM preprocessing files saved.


In [48]:
# Chapter 7.9 — Build and freeze Final Outer-Train GCCM causal weights

from pathlib import Path
import numpy as np
import pandas as pd

final_diag_path = (
    Path(GCCM_OUTPUT_DIR)
    / "seattle_gccm_outer_train_convergence_diagnostics.csv"
)

final_diag = pd.read_csv(final_diag_path)

# Robust boolean restoration
if final_diag["converged"].dtype == object:
    final_diag["converged"] = (
        final_diag["converged"]
        .astype(str)
        .str.lower()
        .map({"true": True, "false": False})
    )
else:
    final_diag["converged"] = (
        final_diag["converged"].astype(bool)
    )

feature_order = [
    "bathrooms",
    "sqft_living",
    "sqft_lot",
    "grade",
    "condition",
    "waterfront",
    "view",
    "age",
]

final_diag["causal_strength"] = np.where(
    final_diag["converged"],
    final_diag["rho_Lmax"],
    0.0
)

strength_sum_final = (
    final_diag["causal_strength"].sum()
)

assert strength_sum_final > 0

final_diag["causal_weight"] = (
    final_diag["causal_strength"]
    / strength_sum_final
)

gccm_weights_outer_final = (
    final_diag
    .set_index("feature")
    .loc[feature_order]
    .reset_index()
)

print("===== FINAL OUTER-TRAIN GCCM WEIGHTS =====")

display(
    gccm_weights_outer_final[
        [
            "feature",
            "rho_L10",
            "rho_Lmax",
            "gain",
            "spearman",
            "p_two_sided",
            "CI_lower",
            "CI_upper",
            "converged",
            "causal_strength",
            "causal_weight",
        ]
    ].round(6)
)

print("\nStrength sum:", strength_sum_final)
print(
    "Weight sum:",
    gccm_weights_outer_final["causal_weight"].sum()
)

assert np.isclose(
    gccm_weights_outer_final["causal_weight"].sum(),
    1.0
)

print("\n✓ Final Outer-Train GCCM causal weights frozen.")

===== FINAL OUTER-TRAIN GCCM WEIGHTS =====


,feature,rho_L10,rho_Lmax,gain,spearman,p_two_sided,CI_lower,CI_upper,converged,causal_strength,causal_weight
0,bathrooms,0.093654,0.143220,0.049566,0.942857,0.000000,0.100922,0.185000,True,0.143220,0.101417
1,sqft_living,0.175639,0.270903,0.095264,0.942857,0.000000,0.230668,0.310214,True,0.270903,0.191833
2,sqft_lot,0.049763,0.171303,0.121540,1.000000,0.000000,0.129337,0.212658,True,0.171303,0.121304
3,grade,0.247137,0.310584,0.063447,0.657143,0.000000,0.271282,0.348851,True,0.310584,0.219932
4,condition,0.014780,0.025398,0.010618,0.657143,0.246266,-0.017539,0.068241,False,0.000000,0.000000
5,waterfront,0.021254,0.164455,0.143201,1.000000,0.000000,0.122401,0.205919,True,0.164455,0.116455
6,view,0.098185,0.296150,0.197965,1.000000,0.000000,0.256492,0.334812,True,0.296150,0.209711
7,age,0.005681,0.055566,0.049885,1.000000,0.011139,0.012679,0.098250,True,0.055566,0.039348



Strength sum: 1.4121816039434276
Weight sum: 1.0

✓ Final Outer-Train GCCM causal weights frozen.


In [50]:
# Chapter 7.10 — Save Final Outer-Train causal weights

from pathlib import Path

final_weights_path = (
    Path(GCCM_OUTPUT_DIR)
    / "seattle_gccm_causal_weights_outer_train_v1.csv"
)

gccm_weights_outer_final.to_csv(
    final_weights_path,
    index=False
)

print("Saved:")
print(final_weights_path)

print("\nFile exists:", final_weights_path.exists())
print("File size:", final_weights_path.stat().st_size, "bytes")

print(
    "\n✓ Final Outer-Train causal weights saved."
)

Saved:
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccm\seattle_gccm_causal_weights_outer_train_v1.csv

File exists: True
File size: 1437 bytes

✓ Final Outer-Train causal weights saved.


In [51]:
# Chapter 7.11 — Compare Inner-Fit and Final Outer-Train GCCM weights

inner_weights_path = (
    Path(GCCM_OUTPUT_DIR)
    / "seattle_gccm_causal_weights_inner_fit_v1.csv"
)

inner_weights = pd.read_csv(
    inner_weights_path
)[
    ["feature", "causal_strength", "causal_weight"]
].rename(
    columns={
        "causal_strength": "inner_strength",
        "causal_weight": "inner_weight",
    }
)

outer_weights = (
    gccm_weights_outer_final[
        ["feature", "causal_strength", "causal_weight"]
    ]
    .rename(
        columns={
            "causal_strength": "outer_strength",
            "causal_weight": "outer_weight",
        }
    )
)

weight_stability = (
    inner_weights
    .merge(
        outer_weights,
        on="feature",
        how="inner"
    )
)

weight_stability["weight_change"] = (
    weight_stability["outer_weight"]
    - weight_stability["inner_weight"]
)

print("===== INNER vs FINAL GCCM WEIGHTS =====")

display(
    weight_stability.round(6)
)

print(
    "\nWeight-vector correlation:",
    np.corrcoef(
        weight_stability["inner_weight"],
        weight_stability["outer_weight"]
    )[0, 1]
)

===== INNER vs FINAL GCCM WEIGHTS =====


,feature,inner_strength,inner_weight,outer_strength,outer_weight,weight_change
0,bathrooms,0.193219,0.158578,0.143220,0.101417,-0.057161
1,sqft_living,0.267773,0.219766,0.270903,0.191833,-0.027933
2,sqft_lot,0.201288,0.165200,0.171303,0.121304,-0.043896
3,grade,0.315271,0.258749,0.310584,0.219932,-0.038817
4,condition,0.089394,0.073367,0.000000,0.000000,-0.073367
5,waterfront,0.000000,0.000000,0.164455,0.116455,0.116455
6,view,0.151501,0.124340,0.296150,0.209711,0.085372
7,age,0.000000,0.000000,0.055566,0.039348,0.039348



Weight-vector correlation: 0.6838744386976202


## Chapter 8 — KNN graph sensitivity analysis

The main GCCM experiment uses a Delaunay point graph.

To evaluate whether the estimated causal structure is sensitive to the
point-neighborhood definition, a symmetric K-nearest-neighbor graph is
used as a robustness analysis.

The number of nearest neighbors is fixed at

$$
k=6,
$$

because the Delaunay graph has an average degree of approximately six
for both Inner Fit and Outer Train.

The KNN graph is symmetrized using the union rule: two sites are treated
as neighbors if either site belongs to the other's six nearest neighbors.

This sensitivity analysis does not replace the frozen Delaunay main
experiment.

In [52]:
# Chapter 8.1 — Build symmetric 6-NN graph on Outer Train

import numpy as np
import pandas as pd

from sklearn.neighbors import NearestNeighbors

K_SENSITIVITY = 6

coords_knn = site_coords_outer.copy()
n_knn_sites = len(coords_knn)

# +1 because each point itself is returned as nearest neighbor
nn_model = NearestNeighbors(
    n_neighbors=K_SENSITIVITY + 1,
    metric="euclidean"
)

nn_model.fit(coords_knn)

distances_knn, indices_knn = (
    nn_model.kneighbors(coords_knn)
)

# Remove self-neighbor in first column
directed_knn = indices_knn[:, 1:]

neighbor_sets_knn_outer = [
    set()
    for _ in range(n_knn_sites)
]

# Symmetric UNION graph:
# connect i-j if either i selects j or j selects i
for i in range(n_knn_sites):
    for j in directed_knn[i]:
        j = int(j)

        if i == j:
            continue

        neighbor_sets_knn_outer[i].add(j)
        neighbor_sets_knn_outer[j].add(i)

degrees_knn_outer = np.array([
    len(x)
    for x in neighbor_sets_knn_outer
])

knn_edge_set_outer = set()

for i, neighs in enumerate(
    neighbor_sets_knn_outer
):
    for j in neighs:
        if i < j:
            knn_edge_set_outer.add(
                (i, j)
            )

print("===== OUTER TRAIN SYMMETRIC 6-NN GRAPH =====")

print("Sites:", n_knn_sites)
print("Undirected edges:", len(knn_edge_set_outer))

display(
    pd.Series(
        degrees_knn_outer
    ).describe(
        percentiles=[0.25, 0.50, 0.75]
    ).to_frame("degree")
)

print(
    "Zero-neighbor sites:",
    np.sum(degrees_knn_outer == 0)
)

print(
    "Adjacency symmetric:",
    all(
        i in neighbor_sets_knn_outer[j]
        for i in range(n_knn_sites)
        for j in neighbor_sets_knn_outer[i]
    )
)

print(
    "Any self-neighbors:",
    any(
        i in neighbor_sets_knn_outer[i]
        for i in range(n_knn_sites)
    )
)

===== OUTER TRAIN SYMMETRIC 6-NN GRAPH =====
Sites: 2088
Undirected edges: 7698


,degree
count,2088.000000
mean,7.373563
std,1.401386
min,6.000000
25%,6.000000
50%,7.000000
75%,8.000000
max,13.000000


Zero-neighbor sites: 0
Adjacency symmetric: True
Any self-neighbors: False


In [53]:
# Chapter 8.2 — KNN Ring 1/2/3 audit

rings_knn_outer = exact_graph_rings(
    neighbor_sets_knn_outer,
    max_ring=3
)

knn_ring_summary = []

for k in [1, 2, 3]:

    sizes = np.array([
        len(r)
        for r in rings_knn_outer[k]
    ])

    knn_ring_summary.append({
        "ring": k,
        "mean_size": sizes.mean(),
        "std_size": sizes.std(ddof=1),
        "min_size": sizes.min(),
        "median": np.median(sizes),
        "max_size": sizes.max(),
        "empty_rings": np.sum(sizes == 0)
    })

knn_ring_summary = pd.DataFrame(
    knn_ring_summary
)

display(
    knn_ring_summary.round(6)
)

overlap12 = overlap13 = overlap23 = 0
self_count = 0

for i in range(n_knn_sites):

    r1 = set(rings_knn_outer[1][i])
    r2 = set(rings_knn_outer[2][i])
    r3 = set(rings_knn_outer[3][i])

    overlap12 += len(r1 & r2)
    overlap13 += len(r1 & r3)
    overlap23 += len(r2 & r3)

    self_count += int(
        i in r1
        or i in r2
        or i in r3
    )

print("Ring1 / Ring2 overlap:", overlap12)
print("Ring1 / Ring3 overlap:", overlap13)
print("Ring2 / Ring3 overlap:", overlap23)
print("Self in any ring:", self_count)

assert (
    knn_ring_summary["empty_rings"]
    == 0
).all()

assert overlap12 == 0
assert overlap13 == 0
assert overlap23 == 0
assert self_count == 0

print(
    "\n✓ Symmetric 6-NN graph supports E = 3."
)

,ring,mean_size,std_size,min_size,median,max_size,empty_rings
0,1,7.373563,1.401386,6,7.0,13,0
1,2,10.504789,3.729083,0,10.0,26,6
2,3,14.456897,5.540811,0,14.0,39,8


Ring1 / Ring2 overlap: 0
Ring1 / Ring3 overlap: 0
Ring2 / Ring3 overlap: 0
Self in any ring: 0


AssertionError: 

In [54]:
# Chapter 8.3 — Diagnose empty KNN rings and connected components

import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Sites with empty Ring 2 / Ring 3
# ---------------------------------------------------------

empty_ring2_sites = [
    i
    for i, r in enumerate(rings_knn_outer[2])
    if len(r) == 0
]

empty_ring3_sites = [
    i
    for i, r in enumerate(rings_knn_outer[3])
    if len(r) == 0
]

print("Empty Ring 2 sites:", len(empty_ring2_sites))
print(empty_ring2_sites)

print("\nEmpty Ring 3 sites:", len(empty_ring3_sites))
print(empty_ring3_sites)


# ---------------------------------------------------------
# 2. Connected-component audit
# ---------------------------------------------------------

def connected_components_from_sets(neighbor_sets):

    n = len(neighbor_sets)
    visited = set()
    components = []

    for start in range(n):

        if start in visited:
            continue

        stack = [start]
        component = set()

        while stack:

            node = stack.pop()

            if node in visited:
                continue

            visited.add(node)
            component.add(node)

            for nb in neighbor_sets[node]:
                if nb not in visited:
                    stack.append(nb)

        components.append(
            sorted(component)
        )

    return components


knn_components = connected_components_from_sets(
    neighbor_sets_knn_outer
)

component_sizes = np.array([
    len(c)
    for c in knn_components
])

print("\n===== CONNECTED COMPONENTS =====")

print("Number of components:", len(knn_components))
print("Largest component:", component_sizes.max())
print("Smallest component:", component_sizes.min())

print("\nComponent-size distribution:")
print(
    pd.Series(component_sizes)
    .value_counts()
    .sort_index()
)


# ---------------------------------------------------------
# 3. Which components contain empty-ring sites?
# ---------------------------------------------------------

site_to_component = {}

for cid, comp in enumerate(knn_components):
    for site in comp:
        site_to_component[site] = cid

problem_sites = sorted(
    set(empty_ring2_sites)
    | set(empty_ring3_sites)
)

problem_info = []

for site in problem_sites:

    cid = site_to_component[site]

    problem_info.append({
        "site": site,
        "component_id": cid,
        "component_size": len(
            knn_components[cid]
        ),
        "degree": len(
            neighbor_sets_knn_outer[site]
        ),
        "ring1": len(
            rings_knn_outer[1][site]
        ),
        "ring2": len(
            rings_knn_outer[2][site]
        ),
        "ring3": len(
            rings_knn_outer[3][site]
        )
    })

problem_info = pd.DataFrame(
    problem_info
)

print("\nProblem-site diagnostics:")
display(problem_info)

Empty Ring 2 sites: 6
[374, 674, 704, 1428, 1798, 2010]

Empty Ring 3 sites: 8
[374, 546, 674, 704, 1428, 1746, 1798, 2010]

===== CONNECTED COMPONENTS =====
Number of components: 3
Largest component: 2046
Smallest component: 8

Component-size distribution:
8       1
34      1
2046    1
Name: count, dtype: int64

Problem-site diagnostics:


,site,component_id,component_size,degree,ring1,ring2,ring3
0,374,2,8,7,7,0,0
1,546,2,8,6,6,1,0
2,674,2,8,7,7,0,0
3,704,2,8,7,7,0,0
4,1428,2,8,7,7,0,0
5,1746,2,8,6,6,1,0
6,1798,2,8,7,7,0,0
7,2010,2,8,7,7,0,0


In [55]:
# Chapter 8.4 — Topology-only search for the minimum valid KNN k

import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

K_MIN = 6
K_MAX = 20

coords_scan = site_coords_outer.copy()
n_scan = len(coords_scan)

# ---------------------------------------------------------
# Precompute nearest neighbors once up to K_MAX
# ---------------------------------------------------------

nn_scan = NearestNeighbors(
    n_neighbors=K_MAX + 1,
    metric="euclidean"
)

nn_scan.fit(coords_scan)

_, all_knn_indices = nn_scan.kneighbors(
    coords_scan
)

# First column is each site itself
all_knn_indices = all_knn_indices[:, 1:]


def build_symmetric_knn_graph(
    all_neighbor_indices,
    k
):
    """
    Symmetric union KNN graph.

    i-j is an edge if either i selects j
    or j selects i among the first k nearest neighbors.
    """

    n = len(all_neighbor_indices)

    neighbor_sets = [
        set()
        for _ in range(n)
    ]

    for i in range(n):

        for j in all_neighbor_indices[i, :k]:

            j = int(j)

            if i == j:
                continue

            neighbor_sets[i].add(j)
            neighbor_sets[j].add(i)

    return neighbor_sets


def component_sizes_from_graph(
    neighbor_sets
):
    components = connected_components_from_sets(
        neighbor_sets
    )

    return sorted(
        [len(c) for c in components],
        reverse=True
    )


knn_topology_rows = []

for k in range(K_MIN, K_MAX + 1):

    neigh_sets = build_symmetric_knn_graph(
        all_knn_indices,
        k
    )

    degrees = np.array([
        len(x)
        for x in neigh_sets
    ])

    components = (
        connected_components_from_sets(
            neigh_sets
        )
    )

    component_sizes = sorted(
        [len(c) for c in components],
        reverse=True
    )

    rings_k = exact_graph_rings(
        neigh_sets,
        max_ring=3
    )

    empty_r1 = sum(
        len(r) == 0
        for r in rings_k[1]
    )

    empty_r2 = sum(
        len(r) == 0
        for r in rings_k[2]
    )

    empty_r3 = sum(
        len(r) == 0
        for r in rings_k[3]
    )

    symmetric = all(
        i in neigh_sets[j]
        for i in range(n_scan)
        for j in neigh_sets[i]
    )

    self_neighbor = any(
        i in neigh_sets[i]
        for i in range(n_scan)
    )

    valid_E3 = (
        len(components) == 1
        and empty_r1 == 0
        and empty_r2 == 0
        and empty_r3 == 0
        and symmetric
        and not self_neighbor
    )

    knn_topology_rows.append({
        "k": k,
        "components": len(components),
        "largest_component": component_sizes[0],
        "smallest_component": component_sizes[-1],
        "mean_degree": degrees.mean(),
        "min_degree": degrees.min(),
        "max_degree": degrees.max(),
        "empty_ring1": empty_r1,
        "empty_ring2": empty_r2,
        "empty_ring3": empty_r3,
        "valid_E3": valid_E3
    })


knn_topology_scan = pd.DataFrame(
    knn_topology_rows
)

display(
    knn_topology_scan.round(6)
)

valid_rows = knn_topology_scan[
    knn_topology_scan["valid_E3"]
]

if len(valid_rows) > 0:

    selected_k = int(
        valid_rows.iloc[0]["k"]
    )

    print(
        "\n✓ Minimum topology-valid k for E=3:",
        selected_k
    )

else:

    selected_k = None

    print(
        "\nNo topology-valid k found between",
        K_MIN,
        "and",
        K_MAX
    )

,k,components,largest_component,smallest_component,mean_degree,min_degree,max_degree,empty_ring1,empty_ring2,empty_ring3,valid_E3
0,6,3,2046,8,7.373563,6,13,0,6,8,False
1,7,2,2080,8,8.513410,7,18,0,8,8,False
2,8,1,2088,2088,9.698276,8,18,0,0,0,True
3,9,1,2088,2088,10.856322,9,19,0,0,0,True
4,10,1,2088,2088,12.016284,10,21,0,0,0,True
5,11,1,2088,2088,13.194444,11,23,0,0,0,True
6,12,1,2088,2088,14.321839,12,24,0,0,0,True
7,13,1,2088,2088,15.471264,13,27,0,0,0,True
8,14,1,2088,2088,16.628352,14,28,0,0,0,True
9,15,1,2088,2088,17.779693,15,30,0,0,0,True



✓ Minimum topology-valid k for E=3: 8


In [56]:
# Chapter 8.5 — Freeze the topology-valid symmetric 8-NN graph

K_KNN_FINAL = 8

neighbor_sets_knn8_outer = build_symmetric_knn_graph(
    all_knn_indices,
    K_KNN_FINAL
)

degrees_knn8_outer = np.array([
    len(x)
    for x in neighbor_sets_knn8_outer
])

knn8_edge_set_outer = set()

for i, neighs in enumerate(
    neighbor_sets_knn8_outer
):
    for j in neighs:
        if i < j:
            knn8_edge_set_outer.add(
                (i, j)
            )

components_knn8 = connected_components_from_sets(
    neighbor_sets_knn8_outer
)

rings_knn8_outer = exact_graph_rings(
    neighbor_sets_knn8_outer,
    max_ring=3
)

empty_knn8 = {
    k: sum(
        len(r) == 0
        for r in rings_knn8_outer[k]
    )
    for k in [1, 2, 3]
}

print("===== FROZEN SYMMETRIC 8-NN GRAPH =====")

print("Sites:", n_sites_outer)
print("Connected components:", len(components_knn8))

print(
    "Undirected edges:",
    len(knn8_edge_set_outer)
)

print(
    "Mean degree:",
    degrees_knn8_outer.mean()
)

print(
    "Min / max degree:",
    degrees_knn8_outer.min(),
    degrees_knn8_outer.max()
)

print(
    "Empty Ring 1 / 2 / 3:",
    empty_knn8[1],
    empty_knn8[2],
    empty_knn8[3]
)

assert len(components_knn8) == 1
assert empty_knn8[1] == 0
assert empty_knn8[2] == 0
assert empty_knn8[3] == 0

print(
    "\n✓ Symmetric 8-NN + E=3 frozen for sensitivity analysis."
)

===== FROZEN SYMMETRIC 8-NN GRAPH =====
Sites: 2088
Connected components: 1
Undirected edges: 10125
Mean degree: 9.698275862068966
Min / max degree: 8 18
Empty Ring 1 / 2 / 3: 0 0 0

✓ Symmetric 8-NN + E=3 frozen for sensitivity analysis.


In [57]:
# Chapter 8.6 — Export frozen symmetric 8-NN adjacency

knn8_edge_rows = []

for i, neighs in enumerate(
    neighbor_sets_knn8_outer
):
    for j in neighs:

        knn8_edge_rows.append({
            "from_site": i + 1,
            "to_site": int(j) + 1
        })

knn8_edges_outer = pd.DataFrame(
    knn8_edge_rows
)

knn8_edges_path = (
    Path(GCCM_OUTPUT_DIR)
    / "seattle_gccm_outer_train_knn8_edges.csv"
)

knn8_edges_outer.to_csv(
    knn8_edges_path,
    index=False
)

topology_scan_path = (
    Path(GCCM_OUTPUT_DIR)
    / "seattle_gccm_outer_train_knn_topology_scan.csv"
)

knn_topology_scan.to_csv(
    topology_scan_path,
    index=False
)

print("Saved:")
print(knn8_edges_path)
print(topology_scan_path)

print(
    "\nDirected adjacency rows:",
    len(knn8_edges_outer)
)

print(
    "Expected:",
    2 * len(knn8_edge_set_outer)
)

print(
    "Unique from-sites:",
    knn8_edges_outer["from_site"].nunique()
)

print(
    "Unique to-sites:",
    knn8_edges_outer["to_site"].nunique()
)

assert (
    len(knn8_edges_outer)
    == 2 * len(knn8_edge_set_outer)
)

assert (
    knn8_edges_outer["from_site"].nunique()
    == 2088
)

assert (
    knn8_edges_outer["to_site"].nunique()
    == 2088
)

print(
    "\n✓ Symmetric 8-NN adjacency exported."
)

Saved:
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccm\seattle_gccm_outer_train_knn8_edges.csv
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccm\seattle_gccm_outer_train_knn_topology_scan.csv

Directed adjacency rows: 20250
Expected: 20250
Unique from-sites: 2088
Unique to-sites: 2088

✓ Symmetric 8-NN adjacency exported.


In [58]:
# Chapter 8.7 — Build KNN8 GCCM causal weights

from pathlib import Path
import numpy as np
import pandas as pd

knn8_diag_path = (
    Path(GCCM_OUTPUT_DIR)
    / "seattle_gccm_knn8_outer_train_convergence_diagnostics.csv"
)

knn8_diag = pd.read_csv(
    knn8_diag_path
)

# Robust boolean restoration
if knn8_diag["converged"].dtype == object:
    knn8_diag["converged"] = (
        knn8_diag["converged"]
        .astype(str)
        .str.lower()
        .map({
            "true": True,
            "false": False
        })
    )
else:
    knn8_diag["converged"] = (
        knn8_diag["converged"]
        .astype(bool)
    )

feature_order = [
    "bathrooms",
    "sqft_living",
    "sqft_lot",
    "grade",
    "condition",
    "waterfront",
    "view",
    "age",
]

knn8_diag["causal_strength"] = np.where(
    knn8_diag["converged"],
    knn8_diag["rho_Lmax"],
    0.0
)

knn8_strength_sum = (
    knn8_diag["causal_strength"].sum()
)

assert knn8_strength_sum > 0

knn8_diag["causal_weight"] = (
    knn8_diag["causal_strength"]
    / knn8_strength_sum
)

gccm_weights_knn8 = (
    knn8_diag
    .set_index("feature")
    .loc[feature_order]
    .reset_index()
)

print("===== KNN8 GCCM CAUSAL WEIGHTS =====")

display(
    gccm_weights_knn8[
        [
            "feature",
            "rho_L10",
            "rho_Lmax",
            "spearman",
            "p_two_sided",
            "CI_lower",
            "CI_upper",
            "converged",
            "causal_strength",
            "causal_weight"
        ]
    ].round(6)
)

print(
    "\nStrength sum:",
    knn8_strength_sum
)

print(
    "Weight sum:",
    gccm_weights_knn8[
        "causal_weight"
    ].sum()
)

assert np.isclose(
    gccm_weights_knn8[
        "causal_weight"
    ].sum(),
    1.0
)

===== KNN8 GCCM CAUSAL WEIGHTS =====


,feature,rho_L10,rho_Lmax,spearman,p_two_sided,CI_lower,CI_upper,converged,causal_strength,causal_weight
0,bathrooms,0.088248,0.212304,1,0.000000,0.170944,0.252917,True,0.212304,0.107385
1,sqft_living,0.157435,0.392578,1,0.000000,0.355652,0.428280,True,0.392578,0.198568
2,sqft_lot,0.054512,0.350824,1,0.000000,0.312614,0.387902,True,0.350824,0.177449
3,grade,0.241191,0.455955,1,0.000000,0.421282,0.489298,True,0.455955,0.230625
4,condition,0.011392,0.064854,1,0.003042,0.021998,0.107473,True,0.064854,0.032804
5,waterfront,0.010933,0.061617,1,0.004875,0.018749,0.104259,True,0.061617,0.031166
6,view,0.070134,0.238368,1,0.000000,0.197470,0.278437,True,0.238368,0.120568
7,age,0.010144,0.200543,1,0.000000,0.158994,0.241384,True,0.200543,0.101436



Strength sum: 1.9770440510451102
Weight sum: 1.0


In [59]:
# Chapter 8.8 — Compare Delaunay and KNN8 GCCM structures

delaunay_weights = (
    gccm_weights_outer_final[
        [
            "feature",
            "causal_strength",
            "causal_weight"
        ]
    ]
    .rename(
        columns={
            "causal_strength":
                "delaunay_strength",
            "causal_weight":
                "delaunay_weight"
        }
    )
)

knn8_weights = (
    gccm_weights_knn8[
        [
            "feature",
            "causal_strength",
            "causal_weight"
        ]
    ]
    .rename(
        columns={
            "causal_strength":
                "knn8_strength",
            "causal_weight":
                "knn8_weight"
        }
    )
)

graph_sensitivity = (
    delaunay_weights
    .merge(
        knn8_weights,
        on="feature"
    )
)

graph_sensitivity[
    "weight_change_knn_minus_del"
] = (
    graph_sensitivity["knn8_weight"]
    -
    graph_sensitivity["delaunay_weight"]
)

display(
    graph_sensitivity.round(6)
)

pearson_weight_corr = np.corrcoef(
    graph_sensitivity["delaunay_weight"],
    graph_sensitivity["knn8_weight"]
)[0, 1]

spearman_weight_corr = (
    graph_sensitivity[
        ["delaunay_weight", "knn8_weight"]
    ]
    .corr(method="spearman")
    .iloc[0, 1]
)

print(
    "\nPearson weight correlation:",
    pearson_weight_corr
)

print(
    "Spearman weight correlation:",
    spearman_weight_corr
)

,feature,delaunay_strength,delaunay_weight,knn8_strength,knn8_weight,weight_change_knn_minus_del
0,bathrooms,0.143220,0.101417,0.212304,0.107385,0.005967
1,sqft_living,0.270903,0.191833,0.392578,0.198568,0.006735
2,sqft_lot,0.171303,0.121304,0.350824,0.177449,0.056145
3,grade,0.310584,0.219932,0.455955,0.230625,0.010693
4,condition,0.000000,0.000000,0.064854,0.032804,0.032804
5,waterfront,0.164455,0.116455,0.061617,0.031166,-0.085289
6,view,0.296150,0.209711,0.238368,0.120568,-0.089143
7,age,0.055566,0.039348,0.200543,0.101436,0.062088



Pearson weight correlation: 0.7150531943662966
Spearman weight correlation: 0.7857142857142857


In [60]:
# Chapter 8.9 — Save KNN8 graph-sensitivity results

knn8_weights_path = (
    Path(GCCM_OUTPUT_DIR)
    / "seattle_gccm_causal_weights_outer_train_knn8_v1.csv"
)

graph_compare_path = (
    Path(GCCM_OUTPUT_DIR)
    / "seattle_gccm_delaunay_vs_knn8_weights_v1.csv"
)

gccm_weights_knn8.to_csv(
    knn8_weights_path,
    index=False
)

graph_sensitivity.to_csv(
    graph_compare_path,
    index=False
)

print("Saved:")
print(knn8_weights_path)
print(graph_compare_path)

print(
    "\n✓ KNN8 GCCM graph-sensitivity weights frozen."
)

Saved:
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccm\seattle_gccm_causal_weights_outer_train_knn8_v1.csv
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\outputs\gccm\seattle_gccm_delaunay_vs_knn8_weights_v1.csv

✓ KNN8 GCCM graph-sensitivity weights frozen.
